# 在 Google Colab 中运行 ComfyUI

使用 Colab GPU 启动 ComfyUI，并通过 [Comfy Desktop](https://github.com/Comfy-Org/Comfy-Desktop) 的 **Remote Connection（远程连接）** 或普通浏览器访问。模型和运行环境保存在 Google Drive 的 `ComfyUI-Colab/` 目录中。

**使用步骤：** 在 Colab 的「更改运行时类型」中选择 GPU，再依次运行 **1 → 2 → 3 → 6-1、6-2 或 6-3**。运行单元格之前先填写参数、设置复选框；只有勾选相应复选框并运行单元格时，相关操作才会执行。

**开始前：** 授权挂载 Google Drive 会开放整个云盘的访问权限。建议使用不含个人数据的专用 Google 账号，只安装可信的模型和自定义节点（见末尾「安全说明」）。请遵守 [Colab 使用条款](https://research.google.com/colaboratory/faq.html)；本项目按付费 Colab 方案使用场景设计。


In [ ]:
#@title 0. 检查更新
#@markdown 检查最新的 ComfyUI 正式版本。勾选后将使用最新版本覆盖第 1 步的 `COMFY_REF`。
USE_LATEST_COMFYUI = False #@param {type:"boolean"}
import re, subprocess
found = subprocess.run(["git", "ls-remote", "--tags", "--refs", "https://github.com/Comfy-Org/ComfyUI.git", "v*"],
                       capture_output=True, text=True, timeout=60)
releases = [tuple(map(int, tag.groups())) for tag in
            (re.search(r"refs/tags/v(\d+)\.(\d+)\.(\d+)$", line) for line in found.stdout.splitlines()) if tag]
if found.returncode or not releases:
    raise RuntimeError("无法获取最新版本，请稍后重试。")
LATEST_COMFY_REF = "v" + ".".join(map(str, max(releases)))
print("ComfyUI 最新版本：", LATEST_COMFY_REF)
# Cell 1 sets COMFY_REF from its form, so it takes the choice made here from this name.
COMFY_REF_OVERRIDE = LATEST_COMFY_REF if USE_LATEST_COMFYUI else ""
if USE_LATEST_COMFYUI:
    COMFY_REF = LATEST_COMFY_REF
    print("已选择最新版本作为 COMFY_REF。请继续运行第 1 步。")
else:
    print("COMFY_REF 将使用第 1 步填写的版本。如需最新版本，请勾选 USE_LATEST_COMFYUI 后重新运行。")


In [ ]:
#@title 1. 基础设置
#@markdown Google Drive 保存目录（相对于「我的云端硬盘」）。请勿让多个运行时同时使用同一个目录。
DRIVE_FOLDER_NAME = "ComfyUI-Colab" #@param {type:"string"}
#@markdown ComfyUI 版本标签；创建新环境或在第 11 步更新时使用。
COMFY_REF = "v0.39.0" #@param {type:"string"}
#@markdown 自动保存环境的间隔（秒，最小 60 秒）。
CHECKPOINT_SECONDS = 120 #@param {type:"integer"}
#@markdown 恢复 Google Drive 中已保存的环境；关闭后使用全新环境。
RESTORE_SAVED_ENVIRONMENT = True #@param {type:"boolean"}
#@markdown 额外安装的系统软件包（可选，多个名称用空格分隔）。
APT_PACKAGES = "" #@param {type:"string"}
#@markdown 模型访问方式：`direct` 按需直接从 Drive 读取（首次加载较慢）；`copy` 在初始化时复制所有模型到 Colab 本地磁盘（需要足够空间）。
MODEL_ACCESS = "direct" #@param ["direct", "copy"]
#@markdown 启用 SageAttention 加速注意力计算（主要在视频模型中收益明显）。若当前 GPU 无法使用，将回退到标准注意力；生成异常时请关闭并在新运行时重试。
USE_SAGE_ATTENTION = True #@param {type:"boolean"}
#@markdown 启用 `--fast fp16_accumulation`，加快部分 fp16 运算。此功能仍是实验性质，可能影响画质；默认关闭，结果异常时请关闭并在新运行时重试。
USE_FP16_ACCUMULATION = False #@param {type:"boolean"}
if globals().get("COMFY_REF_OVERRIDE"):
    COMFY_REF = COMFY_REF_OVERRIDE
print("COMFY_REF:", COMFY_REF)


In [ ]:
#@title 2. 解包运行环境代码
import importlib, shutil, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp>=3.11,<4", "requests", "PyJWT[crypto]>=2.8,<3"], check=True)
FILES = {'__init__.py': '"""The code behind the ComfyUI instance on Colab notebooks.\n\nThe notebook\'s cell 2 writes this package to the Colab machine and imports it; the names below\nare what the cells call. Everything runs as root in the notebook\'s kernel except what the\napplication user runs: ComfyUI, pip and custom nodes (commands.app_command) and the file\noperations in its tree (app_tree.app_action).\n\nModules, roughly in the order a runtime uses them:\n  paths         where things are kept (BASE: the application user\'s; PRIVATE: root only)\n  messages      t(): the texts shown, in the notebook\'s language\n  filesystem    small file helpers\n  commands      running commands, as root or as the application user\n  app_tree      the application user\'s tree and the file operations in it\n  dependencies  apt, the virtual environment, Python packages, SageAttention\n  drive         Google Drive and its models folder\n  environment   saving the environment to Drive in parts, and restoring it\n  installation  setup (cell 3) and updating ComfyUI (cell 11)\n  downloads     model downloads (cell 4)\n  cloudflare    cloudflared, the named tunnel and Cloudflare Access (6-3)\n  pairing       pairing codes for the public URL (6-2)\n  proxy         the proxy in front of ComfyUI\n  stop_button   the stop button on ComfyUI\'s page\n  service       starting and stopping (cells 6 and 10)\n  diagnosis     connection diagnosis (cell 7)\n  __main__      the processes the package starts of itself (python -m)\n\nThe modules refer to each other as module.name (paths.BASE, commands.run): the app action and\nthe tests replace such names on their module, which reaches every caller.\n"""\n\n# Every module, so that runtime.<module> is there for whoever has the package.\nfrom . import (app_tree, cloudflare, commands, dependencies, diagnosis, downloads, drive, environment,  # noqa: F401\n               filesystem, installation, messages, pairing, paths, proxy, service, stop_button)\nfrom .downloads import fetch_model, model_token_name\nfrom .drive import hide_drive_models, sync_models\nfrom .environment import checkpoint\nfrom .installation import setup, update_comfyui\nfrom .pairing import issue_pairing_code\nfrom .service import (keep_running, load_config, redact, run_diagnosis, start, start_with_access, stop,\n                      zip_outputs)\n\n__all__ = ["checkpoint", "fetch_model", "hide_drive_models", "issue_pairing_code", "keep_running", "load_config",\n           "model_token_name", "redact", "run_diagnosis", "setup", "start", "start_with_access", "stop", "sync_models",\n           "update_comfyui", "zip_outputs"]\n', '__main__.py': '"""The processes the runtime starts of itself: python -m <this package> ACTION."""\n\nimport argparse\nimport asyncio\nfrom pathlib import Path\nimport sys\n\nfrom . import app_tree, commands, diagnosis, paths, proxy, service\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("action", choices=["proxy", "supervise", "diagnose", "app"])\n    parser.add_argument("rest", nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n    if args.action == "app":\n        if commands.drops_privileges():\n            raise SystemExit("app actions must run as the application user")\n        paths.BASE = Path(args.rest[0])\n        app_tree.app_tree_action(args.rest[1], args.rest[2:], sys.stdin.buffer, sys.stdout.buffer)\n    elif args.action == "proxy":\n        from aiohttp import web\n        web.run_app(proxy.create_proxy(service.load_config()), host="127.0.0.1", port=8189, access_log=None)\n    elif args.action == "diagnose":\n        asyncio.run(diagnosis.diagnose(service.load_config()))\n    else:\n        service.supervise(service.load_config())\n', 'app_tree.py': '"""The application user\'s working tree (BASE) and the file operations root has done in it."""\n\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport tarfile\nimport tempfile\n\nfrom . import commands, environment, filesystem, paths\nfrom .messages import t\n\n\ndef create_app_tree():\n    """Make the application user\'s working directories. Root hands over only what it has just\n    made and never follows a link: on a rerun after a failed setup, the tree may already hold\n    whatever that user\'s code left there."""\n    import pwd\n    account = pwd.getpwnam(commands.APP_USER)\n    created = []\n    for directory in (paths.BASE, *(paths.BASE / name for name in ("environment", "models", "output", "temp", "input", "home", "database"))):\n        try:\n            os.mkdir(directory)\n        except FileExistsError:\n            continue\n        created.append(directory)\n    for directory in reversed(created):  # BASE last, so nothing can be swapped underneath meanwhile.\n        os.chown(directory, account.pw_uid, account.pw_gid, follow_symlinks=False)\n\n\ndef app_tree_action(action, args, source, sink):\n    """File operations inside BASE. They run as the application user, so a link planted there\n    by a custom node cannot redirect root to Drive, the keys or system files."""\n    envdir = paths.BASE / "environment"\n    repo = envdir / "ComfyUI"\n    if action == "put":\n        target = filesystem.contained_path(paths.BASE, args[0])\n        target.parent.mkdir(parents=True, exist_ok=True)\n        fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".part", dir=target.parent)\n        try:\n            with os.fdopen(fd, "wb") as file:\n                shutil.copyfileobj(source, file, 4 * 1024 * 1024)\n            if len(args) > 1:\n                os.utime(temporary, ns=(int(args[1]), int(args[1])))\n            os.replace(temporary, target)\n        finally:\n            Path(temporary).unlink(missing_ok=True)\n    elif action == "get":\n        with filesystem.contained_path(paths.BASE, args[0]).open("rb") as file:\n            shutil.copyfileobj(file, sink, 4 * 1024 * 1024)\n    elif action == "prepare":\n        # Keep the core default and Manager\'s direct models path on the same storage.\n        for name in paths.MODEL_FOLDERS:\n            (paths.BASE / "models" / name).mkdir(parents=True, exist_ok=True)\n        repo_models = repo / "models"\n        if repo_models.exists() and not repo_models.is_symlink():\n            # What is already there stays: it may be a link into the read-only view of Drive.\n            shutil.copytree(repo_models, paths.BASE / "models", dirs_exist_ok=True,\n                            copy_function=lambda source, target: os.path.lexists(target) or shutil.copy2(source, target))\n            shutil.rmtree(repo_models)\n        if not repo_models.exists():\n            repo_models.symlink_to(paths.BASE / "models", target_is_directory=True)\n        model_cfg = {"base_path": str(paths.BASE / "models"), "is_default": True, **{name: name for name in paths.MODEL_FOLDERS}}\n        (paths.BASE / "models.yaml").write_text(json.dumps({"colab_drive_cache": model_cfg}))  # JSON is valid YAML.\n        for name in ("user", "custom_nodes"):\n            (repo / name).mkdir(exist_ok=True)\n        if (repo / "user/comfyui.db").exists():\n            shutil.copyfile(repo / "user/comfyui.db", paths.BASE / "database/comfyui.db")\n    elif action == "restore":\n        fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".tar.gz", dir=paths.BASE)\n        try:\n            with os.fdopen(fd, "wb") as file:\n                shutil.copyfileobj(source, file, 4 * 1024 * 1024)\n            environment.restore_archive(temporary, envdir)\n        finally:\n            Path(temporary).unlink(missing_ok=True)\n    elif action == "backup-database":\n        # SQLite\'s online backup includes committed WAL changes consistently.\n        import sqlite3\n        with sqlite3.connect(f"file:{paths.BASE / \'database/comfyui.db\'}?mode=ro", uri=True) as database:\n            with sqlite3.connect(repo / "user/comfyui.db") as target:\n                database.backup(target)\n    elif action == "archive":\n        # One part of the environment, as component_of() divides it.\n        component = args[0]\n\n        def without_other_parts(member):\n            parts = PurePosixPath(member.name).parts\n            if parts[:2] == ("ComfyUI", "user") or (member.isdir() and len(parts) == 3 and parts[1] == "custom_nodes"):\n                return None\n            return environment.archive_filter(member)\n        import gzip\n        # No time of packing in the archive: the same files give the same bytes, and so the same name on Drive.\n        with gzip.GzipFile(fileobj=sink, mode="wb", compresslevel=1, mtime=0) as packed, tarfile.open(\n                fileobj=packed, mode="w|") as tf:\n            if component == "core":\n                tf.add(repo, arcname="ComfyUI", filter=without_other_parts)\n            elif component == "user":\n                tf.add(repo / "user", arcname="ComfyUI/user", filter=environment.archive_filter)\n            elif component == "state":\n                for child in sorted(envdir.iterdir()):\n                    if child.name != "ComfyUI":\n                        tf.add(child, arcname=child.name, filter=environment.archive_filter)\n            elif component.startswith("node/") and component[5:] not in ("", ".", "..") and "/" not in component[5:]:\n                tf.add(repo / "custom_nodes" / component[5:], arcname="ComfyUI/custom_nodes/" + component[5:],\n                       filter=environment.archive_filter)\n            else:\n                raise ValueError(component)\n    elif action == "link-models":\n        view, cache = Path(args[0]), paths.BASE / "models"\n        for parent, _, files in os.walk(cache, followlinks=False):\n            for name in files:\n                link = Path(parent) / name\n                # No longer on Drive: the link into the view leads nowhere.\n                if link.is_symlink() and view in Path(os.readlink(link)).parents and not link.exists():\n                    link.unlink()\n        linked = 0\n        for file in filesystem.regular_files(view):\n            link = cache / file.relative_to(view)\n            if link.parent.is_symlink() or (link.parent.exists() and not link.parent.is_dir()):\n                continue\n            link.parent.mkdir(parents=True, exist_ok=True)\n            if link.is_symlink() and os.readlink(link) != str(file):\n                link.unlink()\n            if not os.path.lexists(link):  # A model of this runtime with the same name stays.\n                link.symlink_to(file)\n                linked += 1\n        sink.write(str(linked).encode())\n    elif action == "zip-output":\n        import zipfile\n        root = paths.BASE / "output"\n        count = 0\n        with zipfile.ZipFile(sink, "w", compression=zipfile.ZIP_STORED) as archive:\n            for parent, dirs, files in os.walk(root, followlinks=False):\n                dirs[:] = [d for d in dirs if not (Path(parent) / d).is_symlink()]\n                for name in files:\n                    file = Path(parent) / name\n                    if file.is_symlink() or not file.is_file():\n                        continue\n                    archive.write(file, file.relative_to(root))\n                    count += 1\n        if not count:\n            raise RuntimeError(t("app.no_outputs"))\n    else:\n        raise ValueError(action)\n\n\ndef app_action(action, *args, stdin=None, stdout=None, limit=commands.APP_OUTPUT_LIMIT, timeout=commands.APP_TIMEOUT):\n    """Perform app_tree_action as the application user. stdin is bytes or a file; stdout is a file."""\n    args = list(map(str, args))\n    if not commands.drops_privileges():\n        import io\n        sink = stdout or io.BytesIO()\n        app_tree_action(action, args, io.BytesIO(stdin) if isinstance(stdin, bytes) else stdin, sink)\n        return b"" if stdout else sink.getvalue()\n    command = commands.app_command(commands.own_command("app", paths.BASE, action, *args))\n    try:\n        code, output, errors = commands.bounded(command, env=commands.application_environment(), stdin=stdin, sink=stdout,\n                                                limit=limit, timeout=timeout, cwd=commands.OWN_DIRECTORY)\n    except RuntimeError as failure:\n        raise RuntimeError(t("app.failed", action=action, failure=failure)) from None\n    if code:\n        detail = errors.decode(errors="replace").strip().splitlines()[-1:]\n        raise RuntimeError(t("app.failed_detail", action=action) + "".join(detail))\n    return output\n\n\ndef receive_file(target, relative, times=None, limit=commands.APP_OUTPUT_LIMIT):\n    """Copy a file out of the application\'s tree to a root-only destination, atomically."""\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".part", dir=target.parent)\n    try:\n        with os.fdopen(fd, "wb") as file:\n            app_action("get", relative, stdout=file, limit=limit)\n        if times:\n            os.utime(temporary, ns=times)\n        os.replace(temporary, target)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n', 'cloudflare.py': '"""cloudflared, and the named tunnel and Cloudflare Access of the custom domain URL (6-3)."""\n\nimport json\nimport os\nimport re\nimport time\nfrom urllib.parse import urlsplit\n\nfrom . import filesystem, paths\nfrom .messages import t\n\n\nCLOUDFLARED_VERSION = "2026.10.0"\n# cloudflared\'s link to Cloudflare. Its default, QUIC, cannot get the UDP buffers it wants in a\n# Colab container ("failed to sufficiently increase receive buffer size"); HTTP/2 runs over TCP.\nTUNNEL_PROTOCOL = "http2"\nCLOUDFLARED_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"\n\n\ndef install_cloudflared():\n    """Download the pinned release and refuse to run anything else."""\n    import requests\n    binary = paths.PRIVATE / "cloudflared"\n    partial = paths.PRIVATE / "cloudflared.part"\n    url = ("https://github.com/cloudflare/cloudflared/releases/download/"\n           + CLOUDFLARED_VERSION + "/cloudflared-linux-amd64")\n    try:\n        with requests.get(url, stream=True, timeout=60) as res:\n            if not res.ok:\n                raise RuntimeError(t("cloudflared.download_failed"))\n            with partial.open("wb") as f:\n                for chunk in res.iter_content(1024 * 1024):\n                    f.write(chunk)\n        if filesystem.file_hash(partial) != CLOUDFLARED_SHA256:\n            raise RuntimeError(t("cloudflared.sha256_mismatch"))\n        partial.chmod(0o700)\n        os.replace(partial, binary)\n    finally:\n        partial.unlink(missing_ok=True)\n\n\ndef access_certs_url(team_domain):\n    return f"https://{team_domain}/cdn-cgi/access/certs"\n\n\ndef access_settings(hostname, audience):\n    """What the notebook was told about the Cloudflare side, checked before anything starts."""\n    hostname = re.sub(r"^https://|/+$", "", hostname.strip().lower())\n    if not re.fullmatch(r"(?=.{4,253}$)([a-z0-9]([a-z0-9-]{0,61}[a-z0-9])?\\.)+[a-z0-9-]{2,63}", hostname):\n        raise ValueError(t("access.invalid_hostname"))\n    audience = (audience or "").strip().lower()\n    if not re.fullmatch(r"[0-9a-f]{64}", audience):\n        raise ValueError(t("access.invalid_audience"))\n    return {"hostname": hostname, "aud": audience}\n\n\ndef access_team_domain(hostname):\n    """The team whose sign-in Cloudflare Access puts in front of the hostname.\n\n    Asked the way a stranger would, over verified HTTPS: only whoever controls the hostname can\n    make it send visitors elsewhere, and then no setting here would help. Without that redirect\n    there is nothing to check signatures against, so nothing is started."""\n    import requests\n    status = "-"\n    for _ in range(4):\n        try:\n            answer = requests.get("https://" + hostname + "/", timeout=10, allow_redirects=False)\n        except requests.RequestException:\n            time.sleep(3)\n            continue\n        sign_in = urlsplit(answer.headers.get("Location", ""))\n        if (answer.status_code in (301, 302, 303, 307, 308) and sign_in.scheme == "https"\n                and re.fullmatch(r"[a-z0-9][a-z0-9-]*\\.cloudflareaccess\\.com", sign_in.netloc)\n                and sign_in.path.startswith("/cdn-cgi/access/login")):\n            return sign_in.netloc\n        status = answer.status_code\n        time.sleep(3)\n    raise RuntimeError(t("access.unprotected", hostname=hostname, status=status))\n\n\ndef tunnel_services(log):\n    """Where the named tunnel sends its hostnames: the services in the latest settings cloudflared\n    reports having received from Cloudflare. None while it has reported none that can be read."""\n    services = None\n    for line in log.splitlines():\n        if "Updated to new configuration" not in line:\n            continue\n        found = re.search(r\'config"?[=:]\\s*"((?:[^"\\\\]|\\\\.)*)"\', line)\n        try:\n            rules = json.loads(json.loads(\'"\' + found.group(1) + \'"\'))["ingress"]\n            services = [str(rule["service"]) for rule in rules]\n        except (AttributeError, ValueError, KeyError, TypeError):\n            services = None\n    return services\n\n\ndef tunnel_leads_to_proxy(services):\n    """Every hostname of the tunnel has to end at the proxy. Any other port of this runtime it\n    named would be served to whoever gets past Cloudflare, without the checks made here."""\n    proxy = [service for service in services if service.rstrip("/") in ("http://127.0.0.1:8189", "http://localhost:8189")]\n    others = [service for service in services if service not in proxy and not re.fullmatch(r"http_status:\\d{3}", service)]\n    return bool(proxy) and not others\n', 'commands.py': '"""Running commands, and running them as the unprivileged application user."""\n\nimport os\nfrom pathlib import Path\nimport signal\nimport stat\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nfrom . import paths\nfrom .messages import t\n\n\nAPP_USER = "comfy"\nSETPRIV = "/usr/bin/setpriv"\n# Bounds on what root accepts from, and how long it waits for, commands run as that user.\nAPP_TIMEOUT = 3600\nAPP_OUTPUT_LIMIT = 16 * 1024 ** 2\n\n\nGPU_ENVIRONMENT_KEYS = (\n    "LD_LIBRARY_PATH", "LD_PRELOAD", "CUDA_VISIBLE_DEVICES", "CUDA_HOME", "CUDA_PATH",\n    "CUDA_MODULE_LOADING", "CUDA_DEVICE_ORDER", "CUDA_CACHE_PATH",\n    "NVIDIA_VISIBLE_DEVICES", "NVIDIA_DRIVER_CAPABILITIES", "PYTORCH_NVML_BASED_CUDA_CHECK",\n)\n\n\ndef application_environment():\n    """Preserve GPU loader/device settings without copying notebook credentials."""\n    env = {"PATH": f"{paths.BASE / \'.venv/bin\'}:/usr/local/bin:/usr/bin:/bin",\n           "HOME": str(paths.BASE / "home"), "LANG": "C.UTF-8",\n           "PYTHONUNBUFFERED": "1", "VIRTUAL_ENV": str(paths.BASE / ".venv"),\n           "PIP_DISABLE_PIP_VERSION_CHECK": "1",\n           # Colab\'s global pip.conf logs to /var/log/pip.log, which only root can write:\n           # as the application user every log record would fail with a traceback.\n           "PIP_CONFIG_FILE": "/dev/null"}\n    for name in GPU_ENVIRONMENT_KEYS:\n        if name in os.environ:\n            env[name] = os.environ[name]\n    if "LD_LIBRARY_PATH" in env:\n        # An empty or relative entry means the current directory, which the application user controls.\n        env["LD_LIBRARY_PATH"] = ":".join(p for p in env["LD_LIBRARY_PATH"].split(":") if p.startswith("/"))\n    return env\n\n\ndef drops_privileges():\n    return sys.platform == "linux" and os.geteuid() == 0\n\n\ndef app_command(args):\n    """Run args as the unprivileged application user; setuid programs cannot give root back."""\n    args = list(map(str, args))\n    if not drops_privileges():\n        return args\n    # Absolute: the application\'s PATH starts in a directory that user can write, and root\n    # is still root while this program is looked up.\n    return [SETPRIV, "--reuid", APP_USER, "--regid", APP_USER, "--init-groups", "--no-new-privs", "--", *args]\n\n\n# The folder this package is in. python -m finds the package in its working directory and puts\n# that directory first on the import path: the processes below start there, never in BASE.\nOWN_DIRECTORY = Path(__file__).resolve().parent.parent\n\n\ndef own_command(action, *args):\n    """This package run as a program (__main__.py). Start it with cwd=OWN_DIRECTORY."""\n    return [sys.executable, "-m", __package__, action, *map(str, args)]\n\n\ndef run(args, *, app=False, capture=False, cwd=None, timeout=None):\n    args = list(map(str, args))\n    env = application_environment() if app else os.environ.copy()\n    # Describe the stage without printing command arguments or credentials.\n    stage = Path(args[0]).name\n    if "-m" in args and args.index("-m") + 1 < len(args):\n        module_index = args.index("-m") + 1\n        stage += " -m " + args[module_index]\n        words = {"pip": 1, "uv": 2}.get(args[module_index], 0)\n        rest = args[module_index + 1:]\n        if rest[:1] == ["--python"]:  # pip aimed at another environment: name the subcommand, not the flag.\n            rest = rest[2:]\n        stage += "".join(" " + word for word in rest[:words])\n    if app:\n        args = app_command(args)\n    if not capture:\n        print(t("command.running") + stage, flush=True)\n    started = time.monotonic()\n    # Child-process file descriptors can bypass Colab\'s notebook output capture.\n    # Always collect output and explicitly display it through notebook Python.\n    if app and drops_privileges():\n        from types import SimpleNamespace\n        try:\n            code, out, err = bounded(args, env=env, cwd=cwd, limit=64 * 1024 ** 2, timeout=timeout or 2 * APP_TIMEOUT)\n        except RuntimeError as failure:\n            raise RuntimeError(t("command.failed", stage=stage, failure=failure)) from None\n        result = SimpleNamespace(returncode=code, stdout=out.decode(errors="replace"), stderr=err.decode(errors="replace"))\n    else:\n        result = subprocess.run(args, env=env, cwd=cwd, capture_output=True, text=True, timeout=timeout)\n    if result.returncode or not capture:\n        if result.stdout:\n            print(result.stdout, end="" if result.stdout.endswith("\\n") else "\\n", flush=True)\n        if result.stderr:\n            print(result.stderr, end="" if result.stderr.endswith("\\n") else "\\n", flush=True)\n    if result.returncode:\n        raise RuntimeError(t("command.exit_code", stage=stage, code=result.returncode))\n    if not capture:\n        print(t("command.done", stage=stage, seconds=time.monotonic() - started), flush=True)\n    return result.stdout if capture else None\n\n\ndef bounded(command, *, env, limit, timeout, stdin=None, sink=None, cwd=None):\n    """Run a command the application user controls and collect what it prints.\n\n    It cannot make root wait past timeout or take more than limit bytes, whether it stalls,\n    floods its output, or leaves a child of its own holding the pipes open.\n    Returns (exit code, output unless sink was given, end of the error output)."""\n    import select\n    with tempfile.TemporaryFile() as spool:\n        if isinstance(stdin, bytes):\n            spool.write(stdin)\n            spool.seek(0)\n            stdin = spool\n        process = subprocess.Popen(command, env=env, cwd=cwd, stdin=stdin or subprocess.DEVNULL,\n                                   stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    deadline = time.monotonic() + timeout\n    streams = {process.stdout.fileno(): True, process.stderr.fileno(): False}\n    output, errors, total, failure = [], b"", 0, None\n    try:\n        while streams and not failure:\n            remaining = deadline - time.monotonic()\n            if remaining <= 0:\n                failure = t("command.timed_out")\n                break\n            ready, _, _ = select.select(list(streams), [], [], min(remaining, 0.5))\n            for fd in ready:\n                chunk = os.read(fd, 1024 * 1024)\n                if not chunk:\n                    del streams[fd]\n                elif not streams[fd]:\n                    errors = (errors + chunk)[-1024 * 1024:]\n                elif total + len(chunk) > limit:\n                    failure = t("command.output_limit")\n                    break\n                else:\n                    total += len(chunk)\n                    sink.write(chunk) if sink else output.append(chunk)\n            if not ready and process.poll() is not None:\n                break  # It exited; anything still holding the pipes is not waited for.\n        if not failure:\n            try:\n                process.wait(timeout=max(deadline - time.monotonic(), 0))\n            except subprocess.TimeoutExpired:\n                failure = t("command.timed_out")\n    finally:\n        if process.poll() is None:\n            process.kill()\n        process.stdout.close()\n        process.stderr.close()\n        process.wait()\n    if failure:\n        raise RuntimeError(failure)\n    return process.returncode, b"".join(output), errors\n\n\ndef stop_app_processes():\n    """End everything the application user still runs, including what a custom node detached\n    from ComfyUI so that it would outlive a stop."""\n    if not drops_privileges():\n        return\n    import pwd\n    try:\n        uid = pwd.getpwnam(APP_USER).pw_uid\n    except KeyError:\n        return\n    # As that user, signal every process it may signal: all of its own, in one step.\n    subprocess.run(app_command(["/usr/bin/kill", "-KILL", "-1"]), env={}, capture_output=True)\n    for _ in range(50):\n        alive = False\n        for entry in Path("/proc").iterdir():\n            try:\n                if entry.name.isdigit() and entry.stat().st_uid == uid:\n                    alive = True\n                    os.kill(int(entry.name), signal.SIGKILL)\n            except (FileNotFoundError, ProcessLookupError):\n                pass\n        if not alive:\n            return\n        time.sleep(0.1)\n    raise RuntimeError(t("user.processes_remain"))\n\n\ndef ensure_app_user():\n    """Create the account ComfyUI, pip and custom nodes run as. It cannot enter PRIVATE (Drive, keys)."""\n    if not Path(SETPRIV).is_file():\n        raise RuntimeError(t("user.setpriv_missing", setpriv=SETPRIV))\n    if subprocess.run(["id", "-u", APP_USER], capture_output=True).returncode:\n        run(["useradd", "--system", "--user-group", "--no-create-home", "--home-dir", paths.BASE / "home",\n             "--shell", "/usr/sbin/nologin", APP_USER])\n    # GPU device nodes are normally open to everyone; join their groups where they are not.\n    for device in [*Path("/dev").glob("nvidia*"), *Path("/dev/dri").glob("*")]:\n        info = device.stat()\n        if stat.S_ISCHR(info.st_mode) and info.st_mode & 0o060 and not info.st_mode & 0o006 and info.st_gid:\n            if subprocess.run(["getent", "group", str(info.st_gid)], capture_output=True).returncode:\n                run(["groupadd", "--gid", info.st_gid, f"{APP_USER}-device-{info.st_gid}"])\n            run(["usermod", "--append", "--groups", info.st_gid, APP_USER])\n', 'dependencies.py': '"""OS packages, the application\'s virtual environment and the Python packages in it."""\n\nimport re\nimport subprocess\nimport sys\n\nfrom . import app_tree, commands, paths\nfrom .messages import t\n\n\n# The release made of Python only: Triton, which PyTorch installs on Linux, compiles its kernels.\nSAGE_ATTENTION_PACKAGE = "sageattention==1.0.6"\n# The call ComfyUI makes, on a small input. The first call compiles the kernels for this GPU.\nSAGE_ATTENTION_PROBE = """\nimport torch\nfrom sageattention import sageattn\nq, k, v = (torch.randn(1, 2, 256, 64, device="cuda", dtype=torch.float16) for _ in range(3))\nout = sageattn(q, k, v, is_causal=False, tensor_layout="HND", sm_scale=None, smooth_k=False)\nassert out.shape == q.shape and bool(torch.isfinite(out).all()), "SageAttention returned an unusable result"\n"""\n\n\ndef installed_apt_packages():\n    """Names dpkg has installed, including names they provide: libglib2.0-0 is provided by\n    libglib2.0-0t64 on current Ubuntu."""\n    try:\n        listing = subprocess.run(["dpkg-query", "-W", "-f", "${db:Status-Abbrev}\\t${Package}\\t${Provides}\\n"],\n                                 capture_output=True, text=True).stdout\n    except OSError:\n        return set()\n    names = set()\n    for line in listing.splitlines():\n        status, _, rest = line.partition("\\t")\n        package, _, provides = rest.partition("\\t")\n        if status.startswith("ii"):\n            names.add(package)\n            names.update(item.split("(")[0].strip() for item in provides.split(",") if item.strip())\n    return names\n\n\nAPT_INDEX_REFRESHED = []\n\n\ndef apt_install(packages):\n    """Install the packages that are missing. Colab ships most of them, and refreshing the\n    package index alone takes longer than everything else here."""\n    missing = sorted(set(packages) - installed_apt_packages())\n    if not missing:\n        return\n    if not APT_INDEX_REFRESHED:\n        commands.run(["apt-get", "update", "-qq"])\n        APT_INDEX_REFRESHED.append(True)\n    commands.run(["apt-get", "install", "-y", "-qq", *missing])\n\n\ndef install_packages(*args):\n    """Install into the application\'s virtual environment with uv. It takes the same indexes\n    and pins as pip but downloads and unpacks the multi-gigabyte PyTorch stack in parallel."""\n    py = paths.BASE / ".venv/bin/python"\n    commands.run([py, "-m", "uv", "pip", "install", "--python", py, *args], app=True)\n\n\ndef create_venv():\n    """Create the application\'s virtual environment with pip and uv in it.\n\n    The notebook\'s own pip installs them, so nothing is needed from apt: installing\n    pythonX.Y-venv there upgrades the interpreter the running kernel was started from."""\n    venv = paths.BASE / ".venv"\n    py = venv / "bin/python"\n    commands.run([sys.executable, "-m", "venv", "--without-pip", venv], app=True, capture=True)\n    try:\n        commands.run([sys.executable, "-m", "pip", "--python", py, "install", "pip", "uv"], app=True)\n    except RuntimeError:\n        print(t("venv.pip_fallback"), flush=True)\n        ensure_venv_support()\n        commands.run([sys.executable, "-m", "venv", venv], app=True, capture=True)\n        commands.run([py, "-m", "pip", "install", "uv"], app=True)\n    return py\n\n\ndef ensure_venv_support():\n    """Install ensurepip for the notebook\'s Python version when it is absent."""\n    import importlib.util\n    if importlib.util.find_spec("ensurepip") is not None:\n        return\n    version = f"{sys.version_info.major}.{sys.version_info.minor}"\n    package = f"python{version}-venv"\n    print(t("venv.ensurepip_missing", version=version, package=package), flush=True)\n    try:\n        apt_install([package])\n    except RuntimeError:\n        raise RuntimeError(t("venv.package_failed", package=package)) from None\n    importlib.invalidate_caches()\n    if importlib.util.find_spec("ensurepip") is None:\n        raise RuntimeError(t("venv.ensurepip_unavailable", package=package, python=sys.executable))\n\n\ndef write_lock():\n    """Record the installed packages, to install the same ones on the next runtime."""\n    py = paths.BASE / ".venv/bin/python"\n    # No editable or local paths: those cannot survive a new Colab VM.\n    packages = checked_lock(commands.run([py, "-m", "pip", "freeze"], app=True, capture=True, timeout=300))\n    lock = "environment/requirements.lock.txt"\n    if not (paths.BASE / lock).exists() or app_tree.app_action("get", lock).decode() != packages:\n        app_tree.app_action("put", lock, stdin=packages.encode())\n\n\nLOCK_LINE = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]*(==[A-Za-z0-9._+!-]+| @ (git\\+)?https://\\S+)")\n\n\ndef checked_lock(text):\n    """Accept only pinned packages: a lock is installed again on every new runtime, so pip\n    options, editable installs and local paths must not ride along in it."""\n    if any(line.strip() and not LOCK_LINE.fullmatch(line) for line in text.splitlines()):\n        raise RuntimeError(t("environment.unpinned_lock"))\n    return text\n\n\ndef checked_torch_index(url):\n    if not isinstance(url, str) or not re.fullmatch(r"https://download\\.pytorch\\.org/whl/cu\\d+", url):\n        raise RuntimeError(t("setup.invalid_torch_mirror"))\n    return url\n\n\ndef unlisted_apt_packages(saved, requested):\n    """Package names recorded with a saved environment that this setup was not asked for."""\n    names = saved.get("apt_packages", []) if isinstance(saved, dict) else []\n    return sorted({name for name in names if isinstance(name, str)\n                   and re.fullmatch(r"[a-z0-9][a-z0-9+.-]*", name)} - set(requested))\n\n\ndef prepare_sage_attention():\n    """Install SageAttention unless the environment has one already, and try it on this GPU.\n\n    Told to use it, ComfyUI exits at start when the package is absent, and logs an error at\n    every attention call when its kernels cannot be compiled. Both are found out here, and\n    ComfyUI is then started without it."""\n    py = paths.BASE / ".venv/bin/python"\n    try:\n        found = commands.run([py, "-c", "import importlib.util; print(importlib.util.find_spec(\'sageattention\') is not None)"],\n                             app=True, capture=True, timeout=60)\n        if found.strip() != "True":  # A restored environment keeps the version it was saved with.\n            install_packages(SAGE_ATTENTION_PACKAGE)\n        commands.run([py, "-c", SAGE_ATTENTION_PROBE], app=True, timeout=600)\n    except RuntimeError:\n        print(t("setup.sage_unavailable"))\n        return False\n    print(t("setup.sage_enabled"))\n    return True\n', 'diagnosis.py': '"""Connection diagnosis (cell 7) and the report on what the application user can reach."""\n\nimport asyncio\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom urllib.parse import quote\n\nfrom . import commands, proxy\nfrom .messages import t\n\n\n# What the frontend awaits before it removes its splash logo.\nSTARTUP_ROUTES = ("/", "/api/features", "/api/users", "/api/settings", "/api/i18n", "/api/extensions",\n                  "/api/object_info", "/api/userdata?dir=workflows&recurse=true&split=false&full_info=true")\n\n\nasync def probe_frontend(base, headers=None, timeout=30):\n    """Request the frontend\'s startup dependencies at base; print each result and return the failure count."""\n    from aiohttp import ClientSession, ClientTimeout, DummyCookieJar, WSMsgType\n    failures = 0\n    compressed = []\n\n    async def fetch(session, route):\n        started = time.monotonic()\n        try:\n            async with session.get(base + route, headers=headers, allow_redirects=False) as res:\n                body = await res.read()\n                compressed.append("Content-Encoding" in res.headers)\n                return res.status, body, time.monotonic() - started\n        except Exception as exc:\n            return type(exc).__name__, b"", time.monotonic() - started\n\n    def healthy(route, status, seconds):\n        missing_is_normal = status == 404 and "userdata" in route\n        return isinstance(status, int) and (status < 400 or missing_is_normal) and seconds < 10\n\n    async with ClientSession(timeout=ClientTimeout(total=timeout), cookie_jar=DummyCookieJar()) as session:\n        files = set()\n        for route in STARTUP_ROUTES:\n            status, body, seconds = await fetch(session, route)\n            ok = healthy(route, status, seconds)\n            failures += not ok\n            print(t("diagnosis.route", mark="OK" if ok else "NG", status=status, seconds=seconds, size=len(body),\n                    route=route.split("?")[0]))\n            if route == "/":\n                files.update("/" + name.decode() for name in re.findall(rb\'(?:src|href)="\\./(assets/[^"]+)"\', body))\n            elif route == "/api/extensions" and status == 200:\n                # The list comes from ComfyUI; only a path can be appended to the address safely.\n                files.update(quote(name) for name in json.loads(body) if isinstance(name, str) and name.startswith("/"))\n        gate = asyncio.Semaphore(16)\n\n        size = 0\n\n        async def fetch_file(route):\n            nonlocal size\n            async with gate:\n                status, body, seconds = await fetch(session, route)\n                size += len(body)\n                # Follow the chunks each script imports, as the browser does.\n                more = {"/assets/" + name.decode() for name in\n                        re.findall(rb"(?:\\./|assets/)([\\w.-]+-[\\w-]{8}\\.(?:js|css))", body)} if route.startswith("/assets/") else set()\n                return route, status, seconds, more\n        del compressed[:]\n        started = time.monotonic()\n        results, pending, listed = [], sorted(files), set(files)\n        while pending:\n            batch = await asyncio.gather(*map(fetch_file, pending))\n            results += [item[:3] for item in batch]\n            found = set().union(*(item[3] for item in batch)) - files\n            files |= found\n            pending = sorted(found)\n        elapsed = time.monotonic() - started\n        # A name found inside a script may be just text; only a listed file must exist.\n        bad = [item for item in results if not healthy(*item) and (item[0] in listed or item[1] != 404)]\n        failures += len(bad)\n        # The browser downloads these before the UI appears, so this time predicts the wait.\n        print(t("diagnosis.assets", mark="NG" if bad else "OK", total=len(results), failed=len(bad),\n                megabytes=size / 1048576, elapsed=elapsed, compressed=sum(compressed)))\n        for route, status, seconds in bad[:10]:\n            print(t("diagnosis.failed_asset", status=status, seconds=seconds, route=route))\n        try:\n            # Browsers always offer compression; the first frame carries the client id.\n            async with session.ws_connect(base.replace("http", "ws", 1) + "/ws", headers=headers, compress=15) as ws:\n                message = await ws.receive(timeout=10)\n                kind = json.loads(message.data).get("type") if message.type == WSMsgType.TEXT else message.type.name\n        except Exception as exc:\n            kind = type(exc).__name__\n        failures += kind != "status"\n        print(t("diagnosis.websocket", mark="OK" if kind == "status" else "NG", kind=kind))\n    return failures\n\n\nasync def diagnose(config):\n    """Find the hop where the UI stops loading: ComfyUI, the auth proxy, or the tunnel."""\n    if "mode" not in config:\n        raise SystemExit(t("diagnosis.not_running"))\n    auth = {"Cookie": proxy.COOKIE + "=" + config["cookie_secret"]} if config.get("cookie_secret") else None\n    targets = [(t("diagnosis.target_comfyui"), "http://127.0.0.1:8188", None)]\n    if config["mode"] != "access":  # There the proxy answers only what Cloudflare Access has signed.\n        targets.append((t("diagnosis.target_proxy"), "http://127.0.0.1:8189", auth))\n    if config.get("remote_url"):\n        targets.append((t("diagnosis.target_public"), config["remote_url"], auth))\n    for label, base, headers in targets:\n        print("== " + label, flush=True)\n        if await probe_frontend(base, headers):\n            print(t("diagnosis.failed_at", label=label))\n            return label\n    # Colab\'s forwarding and Cloudflare Access only answer a signed-in browser, so they cannot be tried from here.\n    if config["mode"] == "colab":\n        print(t("diagnosis.colab_ok"))\n    elif config["mode"] == "access":\n        print(t("diagnosis.access_ok"))\n    else:\n        print(t("diagnosis.all_ok"))\n    return None\n\n\ndef isolation_report():\n    """List the local services the application user can reach. Colab runs its own services as\n    root; one that answers without authentication would let a custom node leave its account."""\n    import socket\n    services = set()\n    for table in ("/proc/net/tcp", "/proc/net/tcp6"):\n        for line in Path(table).read_text().splitlines()[1:]:\n            fields = line.split()\n            address, port = fields[1].rsplit(":", 1)\n            if fields[3] != "0A":  # Not listening.\n                continue\n            host = socket.inet_ntoa(bytes.fromhex(address)[::-1]) if len(address) == 8 else "127.0.0.1"\n            services.add(("127.0.0.1" if host == "0.0.0.0" else host, int(port, 16)))\n    for name, value in sorted(os.environ.items()):\n        match = re.fullmatch(r"(\\d+\\.\\d+\\.\\d+\\.\\d+):(\\d+)", value)\n        if match and re.search(r"CRED|TBE|METADATA", name):\n            services.add((match.group(1), int(match.group(2))))\n    services.add(("169.254.169.254", 80))\n    # Unix sockets that accept connections: by path, or "@name" in the abstract namespace.\n    sockets = sorted({fields[7] for fields in map(str.split, Path("/proc/net/unix").read_text().splitlines()[1:])\n                      if len(fields) > 7 and fields[3] == "00010000"})\n    probe = ("import json, socket, sys\\n"\n             "def ask(host, port, path):\\n"\n             "    with socket.create_connection((host, port), 3) as s:\\n"\n             "        try:\\n"\n             "            s.sendall((\'GET \' + path + \' HTTP/1.0\\\\r\\\\nHost: localhost\\\\r\\\\n\\\\r\\\\n\').encode())\\n"\n             "            return s.recv(64).split(b\'\\\\r\\\\n\')[0].decode(\'latin-1\')\\n"\n             "        except OSError:\\n"\n             "            return \'\'\\n"\n             "tcp, unix = json.load(sys.stdin)\\n"\n             "for host, port in tcp:\\n"\n             "    try:\\n"\n             "        answer = ask(host, port, \'/\') or \'connected, no HTTP reply\'\\n"\n             "    except OSError as exc:\\n"\n             "        answer = \'unreachable (\' + type(exc).__name__ + \')\'\\n"\n             "    if answer.startswith(\'HTTP/\'):\\n"\n             "        try:\\n"\n             "            answer += \' | /api/kernels: \' + ask(host, port, \'/api/kernels\').partition(\' \')[2]\\n"\n             "        except OSError:\\n"\n             "            pass\\n"\n             "    print(f\'tcp\\\\t{host}:{port}\\\\t{answer}\', flush=True)\\n"\n             "for name in unix:\\n"\n             "    s = socket.socket(socket.AF_UNIX)\\n"\n             "    s.settimeout(3)\\n"\n             "    try:\\n"\n             "        s.connect(\'\\\\0\' + name[1:] if name.startswith(\'@\') else name)\\n"\n             "        print(f\'unix\\\\t{name}\\\\tconnected\', flush=True)\\n"\n             "    except OSError:\\n"\n             "        pass\\n"\n             "    s.close()\\n")\n    result = subprocess.run(commands.app_command([sys.executable, "-c", probe]), env=commands.application_environment(),\n                            input=json.dumps([sorted(services), sockets]), capture_output=True, text=True, timeout=600)\n    owners = subprocess.run(["ss", "-H", "-ltnp"], capture_output=True, text=True).stdout if shutil.which("ss") else ""\n    print(t("isolation.heading", user=commands.APP_USER))\n    reachable = []\n    for line in result.stdout.splitlines():\n        kind, _, rest = line.partition("\\t")\n        address, _, answer = rest.partition("\\t")\n        if kind == "unix":\n            reachable.append(address)\n            continue\n        port = address.rsplit(":", 1)[1]\n        owner = sorted(set(re.findall(r\'\\(\\("([^"]+)"\', " ".join(\n            row for row in owners.splitlines() if row.split()[3].rsplit(":", 1)[-1] == port))))\n        note = {"8188": t("isolation.comfyui"), "8189": t("isolation.proxy")}.get(port, ", ".join(owner) or t("common.unknown"))\n        print(f"  {address:<24} {answer:<56} {note}")\n    print(t("isolation.unix_sockets", listening=len(sockets), reachable=len(reachable)), ", ".join(reachable) or t("common.none"))\n    import ctypes\n    abi = ctypes.CDLL(None, use_errno=True).syscall(444, None, 0, 1)  # landlock_create_ruleset(version)\n    lsm = Path("/sys/kernel/security/lsm")\n    print(t("isolation.kernel"), os.uname().release, "| Landlock ABI:", abi if abi > 0 else t("common.none"),\n          "| LSM:", lsm.read_text().strip() if lsm.exists() else t("common.unknown"))\n    effective = int(re.search(r"CapEff:\\s*([0-9a-f]+)", Path("/proc/self/status").read_text()).group(1), 16)\n    print(t("isolation.root_capabilities"), ", ".join(f"{name}={t(\'common.yes\') if effective >> bit & 1 else t(\'common.no\')}" for name, bit in\n                               (("NET_ADMIN", 12), ("SYS_ADMIN", 21), ("SETUID", 7), ("SYS_PTRACE", 19))))\n\n    def usable(command):\n        if not shutil.which(command[0]):\n            return t("isolation.no_command")\n        return t("isolation.available") if not subprocess.run(command, capture_output=True).returncode else t("isolation.unavailable")\n    print("iptables:", usable(["iptables", "-S", "OUTPUT"]), "| network namespace:", usable(["unshare", "--net", "true"]),\n          "| mount namespace:", usable(["unshare", "--mount", "true"]), "| ip:", t("common.present") if shutil.which("ip") else t("common.none"),\n          "| /dev/net/tun:", t("common.present") if Path("/dev/net/tun").exists() else t("common.none"))\n', 'downloads.py': '"""Downloading models from Hugging Face and Civitai (cell 4)."""\n\nimport hashlib\nimport hmac\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nfrom urllib.parse import urlsplit, urljoin, quote, unquote, parse_qs\n\nfrom . import app_tree, drive, filesystem, paths\nfrom .messages import t\n\n\ndef model_download_url(source, *, hf_repo="", hf_file="", hf_revision="main", civitai_version=""):\n    if source == "Hugging Face":\n        if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", hf_repo):\n            raise ValueError(t("download.hf_repository"))\n        file = filesystem.safe_relative(hf_file)\n        if not hf_revision or "\\x00" in hf_revision:\n            raise ValueError(t("download.hf_revision"))\n        return f"https://huggingface.co/{hf_repo}/resolve/{quote(hf_revision, safe=\'\')}/{quote(file, safe=\'/\')}"\n    if source == "Civitai":\n        if not re.fullmatch(r"[1-9][0-9]*", str(civitai_version)):\n            raise ValueError(t("download.civitai_version"))\n        return "https://civitai.com/api/download/models/" + str(civitai_version)\n    raise ValueError(t("download.unknown_source"))\n\n\ndef parse_model_url(url):\n    """Download URL and file name for a Hugging Face or Civitai link copied from a browser."""\n    parsed = urlsplit(url.strip())\n    parts = [unquote(part) for part in parsed.path.split("/") if part]\n    host = (parsed.hostname or "").removeprefix("www.")\n    if parsed.scheme == "https" and host == "huggingface.co":\n        if len(parts) < 5 or parts[2] not in {"resolve", "blob"}:\n            raise ValueError(t("download.hf_url"))\n        return model_download_url("Hugging Face", hf_repo="/".join(parts[:2]), hf_file="/".join(parts[4:]),\n                                  hf_revision=parts[3]), parts[-1]\n    if parsed.scheme == "https" and host == "civitai.com":\n        version = parse_qs(parsed.query).get("modelVersionId", [""])[0]\n        if parts[:3] == ["api", "download", "models"] and len(parts) == 4:\n            version = parts[3]\n        elif parts[:1] != ["models"] or not version:\n            raise ValueError(t("download.civitai_url"))\n        return model_download_url("Civitai", civitai_version=version), ""\n    raise ValueError(t("download.unsupported_url"))\n\n\ndef model_token_name(url):\n    """The Colab secret holding the access token for the site a model link points at."""\n    return "CIVITAI_TOKEN" if "civitai.com" in (urlsplit(url.strip()).hostname or "") else "HF_TOKEN"\n\n\ndef served_filename(headers):\n    """The file name a download response declares, or "" unless it is a plain name."""\n    from email.message import EmailMessage\n    message = EmailMessage()\n    try:\n        message["Content-Disposition"] = headers.get("Content-Disposition") or "attachment"\n        name = message.get_filename() or ""\n    except (ValueError, TypeError, LookupError):\n        return ""\n    name = name.replace("\\\\", "/").rsplit("/", 1)[-1]\n    return name if re.fullmatch(r"[^\\x00-\\x1f:]{1,200}", name) and not name.startswith(".") else ""\n\n\ndef fetch_model(config, url, folder, filename="", sha256="", token_secret="", runtime_only=False):\n    """Download the model a pasted link points at into a folder of Drive\'s models directory,\n    or with runtime_only into the same folder of this runtime\'s models, leaving Drive alone."""\n    download, suggested = parse_model_url(url)\n    folder = filesystem.safe_relative(folder.strip().strip("/"))\n    filename = filename.strip() or suggested\n    if "/" in filename:\n        raise ValueError(t("download.filename_has_folder"))\n    if filename:\n        return download_model(config, download, folder + "/" + filename, sha256.strip(), token_secret,\n                              runtime_only=runtime_only)\n    return download_model(config, download, folder, sha256.strip(), token_secret, name_from_server=True,\n                          runtime_only=runtime_only)\n\n\ndef checked_https_url(url):\n    parsed = urlsplit(url)\n    if parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password or parsed.port not in {None, 443}:\n        raise ValueError(t("download.not_https"))\n    import ipaddress\n    import socket\n    if parsed.hostname in {"localhost", "metadata.google.internal"}:\n        raise ValueError(t("download.local_redirect"))\n    try:\n        addresses = {ipaddress.ip_address(parsed.hostname)}\n    except ValueError:\n        # A public-looking name can still point at the VM\'s internal services.\n        try:\n            found = socket.getaddrinfo(parsed.hostname, 443, type=socket.SOCK_STREAM)\n        except OSError:\n            raise ValueError(t("download.unresolved_host")) from None\n        addresses = {ipaddress.ip_address(item[4][0].split("%")[0]) for item in found}\n    if any(not ip.is_global for ip in addresses):\n        raise ValueError(t("download.private_redirect"))\n    return parsed\n\n\nclass PlainProgress:\n    """Download progress as lines of text, where the notebook\'s own bar is not available."""\n    def __init__(self, total):\n        self.total, self.done, self.shown = total, 0, 0\n\n    def update(self, amount):\n        self.done += amount\n        megabytes = self.done / 1048576\n        if self.total:\n            step = self.done * 10 // self.total\n            line = t("download.progress", done=megabytes, total=self.total / 1048576, percent=self.done * 100 / self.total)\n        else:\n            step = self.done // (512 * 1048576)\n            line = t("download.progress_size", done=megabytes)\n        if step != self.shown:\n            self.shown = step\n            print(line, flush=True)\n\n    def close(self):\n        pass\n\n\ndef download_progress(total):\n    """A progress bar in the notebook\'s output for a download of total bytes (0: unknown)."""\n    try:\n        from tqdm.auto import tqdm\n    except ImportError:\n        return PlainProgress(total)\n    return tqdm(total=total or None, desc=t("download.label"), unit="B", unit_scale=True, unit_divisor=1024)\n\n\ndef stream_model(session, url, target, token_secret=""):\n    """Follow HTTPS redirects, forwarding the token only to the original host."""\n    initial_host = checked_https_url(url).hostname\n    current = url\n    for _ in range(11):\n        parsed = checked_https_url(current)\n        headers = {"User-Agent": "ComfyUI-Colab-Downloader/1"}\n        if token_secret and parsed.hostname == initial_host:\n            headers["Authorization"] = "Bearer " + token_secret\n        with session.get(current, headers=headers, stream=True, timeout=(30, 300), allow_redirects=False) as response:\n            if response.status_code in {301, 302, 303, 307, 308}:\n                location = response.headers.get("Location")\n                if not location:\n                    raise RuntimeError(t("download.redirect_without_target"))\n                current = urljoin(current, location)\n                continue\n            if response.status_code != 200:\n                raise RuntimeError(t("download.http_error", status=response.status_code))\n            content_type = response.headers.get("Content-Type", "").lower()\n            if "text/html" in content_type or "application/json" in content_type:\n                raise RuntimeError(t("download.not_a_model"))\n            digest = hashlib.sha256()\n            length = str(response.headers.get("Content-Length", ""))\n            progress = download_progress(int(length) if length.isdigit() else 0)\n            try:\n                with Path(target).open("wb") as file:\n                    for chunk in response.iter_content(4 * 1024 * 1024):\n                        file.write(chunk)\n                        digest.update(chunk)\n                        progress.update(len(chunk))\n            finally:\n                progress.close()\n            if not Path(target).stat().st_size:\n                raise RuntimeError(t("download.empty"))\n            return digest.hexdigest(), served_filename(response.headers)\n    raise RuntimeError(t("download.too_many_redirects"))\n\n\ndef keep_on_runtime(relative):\n    """Mark a model of this runtime as not to be saved: saving sends its new models to Drive."""\n    origin_file = paths.PRIVATE / "model-origins.json"\n    origins = set(json.loads(origin_file.read_text())) if origin_file.exists() else set()\n    filesystem.private_write(origin_file, json.dumps(sorted(origins | {relative})))\n\n\ndef download_model(config, url, relative_path, sha256="", token_secret="", name_from_server=False, runtime_only=False):\n    """Save a model under relative_path in Drive\'s models folder. With name_from_server,\n    relative_path is the folder and the file takes the name the site serves it under.\n    With runtime_only it goes into this runtime\'s models folder instead and never to Drive."""\n    import requests\n    rel = filesystem.safe_relative(relative_path)\n    parsed = checked_https_url(url)\n    if parsed.hostname not in {"huggingface.co", "civitai.com"}:\n        raise ValueError(t("download.host_not_allowed"))\n    if parsed.query or parsed.fragment:\n        raise ValueError(t("download.url_has_query"))\n    if sha256 and not re.fullmatch(r"[a-fA-F0-9]{64}", sha256):\n        raise ValueError(t("download.invalid_sha256"))\n    if runtime_only:\n        # Checked and completed where only root can go, then handed to the application user.\n        folder = paths.PRIVATE / "downloads"\n        if not name_from_server and os.path.lexists(paths.BASE / "models" / rel):\n            raise FileExistsError(t("download.exists_runtime"))\n    else:\n        models = drive.drive_root(config) / "models"\n        models.mkdir(exist_ok=True)\n        target = filesystem.contained_path(models, rel)\n        if not name_from_server and target.exists():\n            raise FileExistsError(t("download.exists"))\n        folder = target if name_from_server else target.parent\n    folder.mkdir(parents=True, exist_ok=True)\n    partial = folder / (".comfy-" + secrets.token_hex(8) + ".part")\n    try:\n        with requests.Session() as session:\n            session.trust_env = False  # No implicit .netrc credentials or proxy settings.\n            digest, served = stream_model(session, url, partial, token_secret)\n        if sha256 and not hmac.compare_digest(digest, sha256.lower()):\n            raise RuntimeError(t("download.sha256_mismatch"))\n        if name_from_server:\n            if not served:\n                raise RuntimeError(t("download.no_filename"))\n            rel = filesystem.safe_relative(rel + "/" + served)\n        if runtime_only:\n            if os.path.lexists(paths.BASE / "models" / rel):\n                raise FileExistsError(t("download.exists_runtime"))\n            with partial.open("rb") as stream:\n                app_tree.app_action("put", "models/" + rel, stdin=stream)\n            keep_on_runtime(rel)\n            print(t("download.saved_runtime"), rel, "SHA256:", digest)\n            return\n        target = filesystem.contained_path(models, rel)\n        if target.exists():\n            raise FileExistsError(t("download.exists_not_overwritten"))\n        os.replace(partial, target)\n        drive.sync_models(config)\n        print(t("download.saved"), rel, "SHA256:", digest)\n    except requests.RequestException:\n        raise RuntimeError(t("download.connection_failed")) from None\n    finally:\n        partial.unlink(missing_ok=True)\n', 'drive.py': '"""Google Drive: mounting it, its models folder and how the runtime\'s models follow it."""\n\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\n\nfrom . import app_tree, commands, dependencies, filesystem, paths\nfrom .messages import t\n\n\ndef mount_drive():\n    """Mount Drive under a root-only parent, out of reach of the application user."""\n    if sys.platform != "linux" or os.geteuid() != 0:\n        raise RuntimeError(t("drive.needs_root"))\n    if paths.PRIVATE.is_symlink() or paths.DRIVE_MOUNT.is_symlink():\n        raise RuntimeError(t("drive.private_symlink"))\n    paths.PRIVATE.mkdir(mode=0o700, exist_ok=True)\n    paths.PRIVATE.chmod(0o700)\n    if paths.PRIVATE.stat().st_uid != 0:\n        raise RuntimeError(t("drive.private_owner"))\n    for line in Path("/proc/mounts").read_text().splitlines():\n        fields = line.split()\n        if len(fields) > 2 and fields[2] == "fuse.drive" and fields[1] != str(paths.DRIVE_MOUNT):\n            raise RuntimeError(t("drive.mounted_elsewhere"))\n    from google.colab import drive\n    drive.mount(str(paths.DRIVE_MOUNT))\n    paths.PRIVATE.chmod(0o700)\n    if not (paths.DRIVE_MOUNT / "MyDrive").is_dir():\n        raise RuntimeError(t("drive.no_mydrive"))\n\n\ndef drive_root(config):\n    mydrive = paths.DRIVE_MOUNT / "MyDrive"\n    if not mydrive.is_dir():\n        raise RuntimeError(t("drive.not_mounted"))\n    return filesystem.contained_path(mydrive, config["drive_folder"])\n\n\ndef drive_models_shown():\n    return os.path.ismount(paths.DRIVE_MODELS)\n\n\ndef show_drive_models(config):\n    """Give the application user a read-only view of Drive\'s models folder, and of nothing else.\n\n    bindfs presents that one folder as the application user\'s own files; the Drive mount it\n    reads from stays where only root can go. The view lies outside the application\'s tree,\n    so that user cannot move the place root mounts on."""\n    source = drive_root(config) / "models"\n    source.mkdir(exist_ok=True)\n    if drive_models_shown():\n        return\n    try:\n        if paths.DRIVE_MODELS.is_symlink():\n            raise RuntimeError(t("models.view_place", path=paths.DRIVE_MODELS))\n        paths.DRIVE_MODELS.mkdir(mode=0o755, exist_ok=True)\n        if paths.DRIVE_MODELS.stat().st_uid != os.geteuid() or any(paths.DRIVE_MODELS.iterdir()):\n            raise RuntimeError(t("models.view_place", path=paths.DRIVE_MODELS))\n        if not shutil.which("bindfs"):\n            if not dependencies.APT_INDEX_REFRESHED:\n                commands.run(["apt-get", "update", "-qq"])\n                dependencies.APT_INDEX_REFRESHED.append(True)\n            # Without what it only recommends: that would swap the FUSE tools Drive is mounted with.\n            commands.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", "bindfs"])\n        commands.run(["bindfs", "-r", "--force-user=" + commands.APP_USER, "--force-group=" + commands.APP_USER, "--perms=a=rX",\n                      "-o", "allow_other", source, paths.DRIVE_MODELS])\n        commands.run(["ls", paths.DRIVE_MODELS], app=True, capture=True)  # The application user can really look in.\n    except (RuntimeError, OSError) as failure:\n        hide_drive_models()\n        raise RuntimeError(t("models.view_failed", reason=failure)) from None\n\n\ndef hide_drive_models():\n    """Take the view away again: Drive cannot be unmounted while something still reads from it."""\n    for command in (["umount", paths.DRIVE_MODELS], ["fusermount", "-u", paths.DRIVE_MODELS], ["umount", "-l", paths.DRIVE_MODELS]):\n        if not drive_models_shown():\n            return\n        try:\n            subprocess.run(list(map(str, command)), capture_output=True, timeout=60)\n        except (OSError, subprocess.TimeoutExpired):\n            pass\n\n\ndef link_models(config):\n    """With the view in place nothing is copied: ComfyUI\'s models folder gets a link to each\n    of Drive\'s files, next to the models downloaded on this runtime."""\n    show_drive_models(config)\n    linked = int(app_tree.app_action("link-models", paths.DRIVE_MODELS) or 0)\n    print(t("models.linked", linked=linked))\n    return linked\n\n\ndef sync_models(config):\n    """Drive is authoritative; refresh the private runtime\'s working copy."""\n    if config.get("model_access") == "direct":\n        return link_models(config)\n    source = drive_root(config) / "models"\n    source.mkdir(exist_ok=True)\n    copied = 0\n    origin_file = paths.PRIVATE / "model-origins.json"\n    origins = set(json.loads(origin_file.read_text())) if origin_file.exists() else set()\n    for file in filesystem.regular_files(source):\n        relative = filesystem.safe_relative(file.relative_to(source).as_posix())\n        origins.add(relative)\n        target = paths.BASE / "models" / relative\n        st = file.stat()\n        if target.exists():\n            old = target.stat()\n            if old.st_size == st.st_size and abs(old.st_mtime - st.st_mtime) < 1:\n                continue\n        with file.open("rb") as stream:\n            app_tree.app_action("put", "models/" + relative, st.st_mtime_ns, stdin=stream)\n        copied += 1\n    filesystem.private_write(origin_file, json.dumps(sorted(origins)))\n    print(t("models.synced", copied=copied))\n    return copied\n\n\ndef save_new_cache_models(config, force=False):\n    # Preserve Manager downloads without overwriting manually updated Drive files.\n    destination = drive_root(config) / "models"\n    destination.mkdir(exist_ok=True)\n    origin_file = paths.PRIVATE / "model-origins.json"\n    origins = set(json.loads(origin_file.read_text())) if origin_file.exists() else set()\n    for file in filesystem.regular_files(paths.BASE / "models"):\n        relative = file.relative_to(paths.BASE / "models").as_posix()\n        if relative in origins:\n            continue\n        st = file.stat()\n        if not force and time.time() - st.st_mtime < 30:\n            continue\n        target = filesystem.contained_path(destination, relative)\n        if not target.exists():\n            # No more than the size root saw: the file is the application user\'s to change.\n            app_tree.receive_file(target, "models/" + relative, (st.st_atime_ns, st.st_mtime_ns), limit=st.st_size)\n        origins.add(relative)\n    filesystem.private_write(origin_file, json.dumps(sorted(origins)))\n', 'environment.py': '"""Saving the environment to Drive in parts and restoring it."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport tarfile\nimport time\n\nfrom . import app_tree, commands, dependencies, drive, filesystem, paths\nfrom .messages import t\n\n\nEXCLUDED = {"models", "output", "temp", "input"}\n\n\nARCHIVE_LIMIT = 4 * 1024 ** 3\n\n\ndef excluded_path(parts):\n    return "__pycache__" in parts or ".venv" in parts or (\n        len(parts) >= 2 and parts[0] == "ComfyUI" and parts[1] in EXCLUDED)\n\n\ndef archive_filter(member):\n    parts = PurePosixPath(member.name).parts\n    if excluded_path(parts) or any(p.endswith(".pyc") for p in parts):\n        return None\n    # Third-party symlinks could capture credentials or files outside the workspace.\n    if member.issym() or member.islnk() or not (member.isfile() or member.isdir()):\n        return None\n    return member\n\n\ndef restore_archive(archive, dest):\n    with tarfile.open(archive, "r:gz") as tf:\n        for member in tf.getmembers():\n            filesystem.safe_relative(member.name)\n            if member.issym() or member.islnk() or not (member.isfile() or member.isdir()):\n                raise ValueError(t("environment.archive_special_files"))\n        if not hasattr(tarfile, "data_filter"):\n            raise RuntimeError(t("environment.archive_needs_data_filter"))\n        tf.extractall(dest, filter="data")\n\n\nGENERATIONS = 3\nPART_NAME = re.compile(r"[a-f0-9]{64}\\.tar\\.gz")\nGENERATION_NAME = re.compile(r"\\d{8}T\\d{6}Z-[a-f0-9]{12}\\.json")\nWHOLE_ARCHIVE_NAME = re.compile(r"\\d{8}T\\d{6}Z-[a-f0-9]{12}\\.tar\\.gz")\n\n\ndef component_of(parts):\n    """The separately saved part of the environment a file belongs to: ComfyUI itself, each\n    custom node, the user\'s data, and the few files kept beside ComfyUI. Saving a workflow\n    then sends the user\'s data, not the custom nodes with it."""\n    if parts[0] != "ComfyUI":\n        return "state"\n    if len(parts) >= 3 and parts[1] == "user":\n        return "user"\n    if len(parts) >= 4 and parts[1] == "custom_nodes":\n        return "node/" + parts[2]\n    return "core"\n\n\ndef component_fingerprints(root):\n    """{component: fingerprint of its files}, from their names, sizes and times."""\n    entries = {}\n    for parent, dirs, files in os.walk(root, followlinks=False):\n        relative = Path(parent).relative_to(root)\n        dirs[:] = sorted(d for d in dirs if not excluded_path((relative / d).parts)\n                         and not (Path(parent) / d).is_symlink())\n        for name in sorted(files):\n            p = Path(parent) / name\n            if p.is_symlink() or name.endswith(".pyc"):\n                continue\n            try:\n                st = p.stat()\n            except FileNotFoundError:\n                continue\n            path = relative / name\n            entries.setdefault(component_of(path.parts), []).append((path.as_posix(), st.st_size, st.st_mtime_ns))\n    return {name: hashlib.sha256(json.dumps(found).encode()).hexdigest() for name, found in entries.items()}\n\n\ndef discard_old_saves(storage):\n    """Keep the newest saves, and of the parts only what one of them needs. Nothing is deleted\n    but files with the names this notebook gives its own."""\n    def named(folder, pattern):\n        return sorted((item for item in folder.iterdir() if not item.is_symlink() and pattern.fullmatch(item.name)),\n                      key=lambda item: item.name, reverse=True)\n    try:\n        current = (storage / "latest.json").read_text()\n        saves = {save: save.read_text() for save in named(storage / "generations", GENERATION_NAME)}\n        # The save Drive points to is kept whatever its name says of its age, then the newest.\n        kept = sorted(saves, key=lambda save: saves[save] != current)[:GENERATIONS]\n        lists = [json.loads(current), *(json.loads(saves[save]) for save in kept)]\n        needed = {part["file"] for save in lists for part in save.get("parts", {}).values()}\n        needed |= {save["archive"] for save in lists if "archive" in save}\n    except (OSError, ValueError, KeyError, TypeError, AttributeError):\n        return  # A save that cannot be read: delete nothing it might need.\n    for old in set(saves) - set(kept):\n        old.unlink()\n    for part in named(storage / "parts", PART_NAME):\n        if part.name not in needed:\n            part.unlink()\n    # Whole-environment archives, from before the parts were saved separately, count as older saves.\n    for old in named(storage, WHOLE_ARCHIVE_NAME)[max(0, GENERATIONS - len(kept)):]:\n        if old.name not in needed:\n            old.unlink()\n\n\ndef checkpoint(config, force=False):\n    """Save what changed. Each part of the environment is its own archive on Drive, named after\n    its content; a save is a short list of the parts it consists of. Only parts that changed\n    since the last save are packed and sent. force: also make sure Drive still has the rest."""\n    import fcntl\n    with (paths.PRIVATE / "checkpoint.lock").open("w") as lock:\n        fcntl.flock(lock, fcntl.LOCK_EX)\n        refuse_unfinished_update()\n        envdir = paths.BASE / "environment"\n        drive.save_new_cache_models(config, force=force)\n        storage = drive.drive_root(config) / "environment"\n        for folder in (storage, storage / "parts", storage / "generations"):\n            folder.mkdir(exist_ok=True)\n        database = paths.BASE / "database/comfyui.db"\n        if database.exists():\n            signature = [(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in\n                         (database, database.with_name(database.name + "-wal")) if p.exists()]\n            signature_text = json.dumps(signature)\n            db_marker = paths.PRIVATE / "database-signature"\n            if force or not db_marker.exists() or db_marker.read_text() != signature_text:\n                app_tree.app_action("backup-database")\n                db_marker.write_text(signature_text)\n        dependencies.write_lock()\n        record = paths.PRIVATE / "saved-parts.json"\n        saved = json.loads(record.read_text()) if record.exists() else {}\n        before = component_fingerprints(envdir)\n        parts, sent = {}, []\n        staging = paths.PRIVATE / "part.tar.gz"\n        for name in sorted(before):\n            known = saved.get(name)\n            if known and known["fingerprint"] == before[name] and (\n                    not force or filesystem.contained_path(storage / "parts", known["file"]).exists()):\n                parts[name] = known\n                continue\n            try:\n                with staging.open("wb") as file:\n                    app_tree.app_action("archive", name, stdout=file, limit=ARCHIVE_LIMIT, timeout=1800)\n                digest = filesystem.file_hash(staging)\n                filesystem.atomic_copy(staging, filesystem.contained_path(storage / "parts", digest + ".tar.gz"))\n            finally:\n                staging.unlink(missing_ok=True)\n            parts[name] = {"fingerprint": before[name], "file": digest + ".tar.gz", "sha256": digest}\n            sent.append(name)\n        if not sent and set(parts) == set(saved):\n            return False\n        after = component_fingerprints(envdir)\n        if any(after.get(name) != before[name] for name in sent):\n            raise RuntimeError(t("environment.busy"))\n        # The list is written last: until then the preceding save is the one Drive points to.\n        save = {"format": 2, "python": config["python"],\n                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n                "parts": {name: {"file": part["file"], "sha256": part["sha256"]} for name, part in parts.items()}}\n        text = json.dumps(save, sort_keys=True)\n        name = (time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-"\n                + hashlib.sha256(text.encode()).hexdigest()[:12] + ".json")\n        filesystem.private_write(paths.PRIVATE / "latest.json", text)\n        filesystem.atomic_copy(paths.PRIVATE / "latest.json", filesystem.contained_path(storage / "generations", name))\n        filesystem.atomic_copy(paths.PRIVATE / "latest.json", filesystem.contained_path(storage, "latest.json"))\n        filesystem.private_write(record, json.dumps(parts))\n        filesystem.private_write(paths.PRIVATE / "checkpoint-status.json", json.dumps(\n            {"ok": True, "saved_at": save["saved_at"], "python": save["python"], "parts": len(parts), "sent": sent}))\n        discard_old_saves(storage)\n        return True\n\n\ndef restore_environment(storage, save):\n    """Bring a saved environment back, part by part (or from the single archive of earlier\n    saves). Returns what this runtime then holds that Drive already has, for the next save."""\n    archive = paths.PRIVATE / "restore.tar.gz"\n\n    def bring(folder, name, digest):\n        filesystem.atomic_copy(filesystem.contained_path(folder, name), archive)\n        if filesystem.file_hash(archive) != digest:\n            raise RuntimeError(t("setup.archive_sha256_mismatch"))\n        with archive.open("rb") as stream:\n            app_tree.app_action("restore", stdin=stream, timeout=2 * commands.APP_TIMEOUT)\n        archive.unlink()\n\n    if "parts" not in save:\n        name = filesystem.safe_relative(save["archive"])\n        if "/" in name or not name.endswith(".tar.gz"):\n            raise RuntimeError(t("setup.invalid_archive_name"))\n        bring(storage, name, save["sha256"])\n        return {}\n    parts = save["parts"]\n    for part in parts.values():\n        if not PART_NAME.fullmatch(str(part["file"])) or part["file"] != str(part["sha256"]) + ".tar.gz":\n            raise RuntimeError(t("setup.invalid_archive_name"))\n    for name in sorted(parts, key=lambda name: (name != "core", name)):  # ComfyUI first, then what goes into it.\n        bring(storage / "parts", parts[name]["file"], parts[name]["sha256"])\n    restored = component_fingerprints(paths.BASE / "environment")\n    return {name: {"fingerprint": restored[name], "file": parts[name]["file"], "sha256": parts[name]["sha256"]}\n            for name in parts if name in restored}\n\n\ndef refuse_unfinished_update():\n    """A ComfyUI source and packages of two different versions must not be saved or started."""\n    if (paths.PRIVATE / "update-incomplete.json").exists():\n        raise RuntimeError(t("update.incomplete"))\n', 'filesystem.py': '"""Small file helpers: hashing, root-only files and paths that must stay inside a folder."""\n\nimport hashlib\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport tempfile\n\nfrom .messages import t\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as file:\n        for chunk in iter(lambda: file.read(4 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef private_write(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True, mode=0o700)\n    path.write_text(data, encoding="utf-8")\n    path.chmod(0o600)\n\n\ndef safe_relative(value):\n    if "\\\\" in value or ":" in value or "\\x00" in value:\n        raise ValueError(t("path.use_slashes"))\n    p = PurePosixPath(value)\n    if not value or p.is_absolute() or any(x in {"..", "."} for x in value.split("/")):\n        raise ValueError(t("path.unsafe"))\n    return str(p)\n\n\ndef contained_path(root, relative):\n    root = Path(root)\n    rel = safe_relative(relative)\n    target = root / rel\n    if root.is_symlink() or not target.resolve().is_relative_to(root.resolve()):\n        raise ValueError(t("path.outside_folder"))\n    current = root\n    for part in PurePosixPath(rel).parts:\n        current /= part\n        if current.is_symlink():\n            raise ValueError(t("path.symlink"))\n    return target\n\n\ndef atomic_copy(source, destination):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".part", dir=destination.parent)\n    os.close(fd)\n    try:\n        shutil.copy2(source, temporary)\n        os.replace(temporary, destination)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\ndef regular_files(root):\n    for parent, dirs, files in os.walk(root, followlinks=False):\n        dirs[:] = [d for d in dirs if not (Path(parent) / d).is_symlink()]\n        for name in files:\n            file = Path(parent) / name\n            if file.is_symlink() or not file.is_file() or name.endswith((".part", ".tmp", ".download")):\n                continue\n            if name.startswith(".comfy-") or not file.stat().st_size:\n                continue  # Empty files are ComfyUI\'s "put_models_here" markers, not models.\n            yield file\n', 'installation.py': '"""Setting up a runtime (cell 3) and updating ComfyUI (cell 11)."""\n\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\nfrom . import app_tree, cloudflare, commands, dependencies, drive, environment, filesystem, paths\nfrom .messages import t\n\n\ndef setup(*, root_name="ComfyUI-Colab", comfy_ref="v0.39.0",\n          checkpoint_seconds=120, apt_packages=(), restore=True, model_access="direct",\n          sage_attention=True, fp16_accumulation=False):\n    if sys.platform != "linux" or not Path("/content").exists() or os.geteuid() != 0:\n        raise RuntimeError(t("setup.not_colab"))\n    if paths.CONFIG.exists():\n        raise RuntimeError(t("setup.already_done"))\n    filesystem.safe_relative(root_name)\n    if not re.fullmatch(r"[A-Za-z0-9._/-]+", comfy_ref) or comfy_ref.startswith("-"):\n        raise ValueError(t("setup.invalid_ref"))\n    if checkpoint_seconds < 60:\n        raise ValueError(t("setup.interval_too_short"))\n    if any(not re.fullmatch(r"[a-z0-9][a-z0-9+.-]*", p) for p in apt_packages):\n        raise ValueError(t("setup.invalid_apt_package"))\n    if subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL).returncode:\n        raise RuntimeError(t("setup.no_gpu"))\n    if paths.PRIVATE.is_symlink():\n        raise RuntimeError(t("drive.private_symlink"))\n    paths.PRIVATE.mkdir(mode=0o700, exist_ok=True)\n    paths.PRIVATE.chmod(0o700)\n    dependencies.apt_install(["git", "ffmpeg", "libgl1", "libglib2.0-0", *apt_packages])\n    drive.mount_drive()\n    if model_access not in ("direct", "copy"):\n        raise ValueError(t("setup.invalid_model_access"))\n    config = {"drive_folder": root_name,\n              "python": f"{sys.version_info.major}.{sys.version_info.minor}",\n              "checkpoint_seconds": checkpoint_seconds, "model_access": model_access,\n              "fp16_accumulation": bool(fp16_accumulation)}\n    storage = drive.drive_root(config)\n    for directory in (storage, storage / "environment", *(storage / "models" / name for name in paths.MODEL_FOLDERS)):\n        directory.mkdir(parents=True, exist_ok=True)\n    # Test the selected folder, not any other location in Drive.\n    probe = storage / "environment/write-probe.txt"\n    probe.write_text("ComfyUI-Colab write test")\n    # The application user owns its working tree from here on. Root reads and writes inside\n    # it only through app_action, because code running there can plant links.\n    envdir = paths.BASE / "environment"\n    commands.ensure_app_user()\n    app_tree.create_app_tree()\n    if model_access == "direct":\n        drive.show_drive_models(config)  # Before the long steps: if it cannot be done, that is known at once.\n    pointer_path = storage / "environment/latest.json"\n    if pointer_path.exists() and not restore:\n        print(t("setup.fresh_environment"))\n    elif pointer_path.exists():\n        pointer = json.loads(pointer_path.read_text())\n        if pointer["python"] != config["python"]:\n            raise RuntimeError(t("setup.python_changed"))\n        filesystem.private_write(paths.PRIVATE / "saved-parts.json", json.dumps(environment.restore_environment(storage / "environment", pointer)))\n        print(t("setup.restored"), pointer["saved_at"])\n        nodes = sorted(item.name for item in (envdir / "ComfyUI/custom_nodes").glob("*")\n                       if item.is_dir() and item.name != "__pycache__")\n        # Restored nodes run again on this runtime; show them so an unwanted one is noticed.\n        print(t("setup.restored_nodes"), ", ".join(nodes) if nodes else t("common.none"))\n    repo = envdir / "ComfyUI"\n    if not repo.exists():\n        commands.run(["git", "clone", "https://github.com/Comfy-Org/ComfyUI.git", repo], app=True)\n        commands.run(["git", "checkout", comfy_ref], app=True, cwd=repo)\n    if not (repo / "main.py").exists():\n        raise RuntimeError(t("setup.source_incomplete"))\n    app_tree.app_action("prepare")\n    previous_meta = (json.loads(app_tree.app_action("get", "environment/runtime.json"))\n                     if (envdir / "runtime.json").exists() else {})\n    # The saved list is a file the application user could rewrite, and apt runs as root:\n    # name what was used before and leave the decision to install it to the notebook\'s setting.\n    wanted = dependencies.unlisted_apt_packages(previous_meta, apt_packages)\n    if wanted:\n        print(t("setup.saved_apt_packages"), " ".join(wanted))\n        print(t("setup.saved_apt_packages_hint"))\n    # Match the CUDA build already supported by the current Colab runtime.\n    # The mirror is persisted because pip freeze can contain +cuXXX versions.\n    lock = envdir / "requirements.lock.txt"\n    locked = lock.exists()\n    if locked:\n        dependencies.checked_lock(app_tree.app_action("get", "environment/requirements.lock.txt").decode())\n        torch_index = previous_meta.get("torch_index_url", "")\n        torch_packages = []\n    else:\n        import importlib.metadata\n        import torch\n        cuda = torch.version.cuda\n        if not cuda or not re.fullmatch(r"\\d+\\.\\d+", cuda):\n            raise RuntimeError(t("setup.no_torch"))\n        torch_index = "https://download.pytorch.org/whl/cu" + cuda.replace(".", "")\n        torch_packages = ["torch==" + importlib.metadata.version("torch"),\n                          "torchvision==" + importlib.metadata.version("torchvision")]\n    dependencies.checked_torch_index(torch_index)\n    app_tree.app_action("put", "environment/runtime.json", stdin=json.dumps(\n        {"apt_packages": sorted(apt_packages), "python": config["python"], "torch_index_url": torch_index}).encode())\n    py = dependencies.create_venv()\n    if locked:\n        # pip looks for each package on every index; uv does only when told to.\n        dependencies.install_packages("--index-strategy", "unsafe-best-match", "--extra-index-url", torch_index, "-r", lock)\n    else:\n        dependencies.install_packages("--index-url", torch_index, *torch_packages)\n        dependencies.install_packages("-r", repo / "requirements.txt", "-r", repo / "manager_requirements.txt")\n        # Explicit requirements are useful for custom nodes added outside Manager.\n        for req in sorted((repo / "custom_nodes").glob("*/requirements.txt")):\n            dependencies.install_packages("-r", req)\n    commands.run([py, "-m", "pip", "check"], app=True)\n    commands.run([py, "-c", "import torch; print(\'PyTorch:\', torch.__version__); print(\'CUDA build:\', torch.version.cuda); torch.cuda.init(); print(\'GPU:\', torch.cuda.get_device_name(0))"], app=True)\n    config["sage_attention"] = bool(sage_attention) and dependencies.prepare_sage_attention()\n    if model_access == "copy":\n        print(t("setup.copying_models"))\n    drive.sync_models(config)\n    cloudflare.install_cloudflared()\n    filesystem.private_write(paths.CONFIG, json.dumps(config))\n    environment.checkpoint(config, force=True)\n    print(t("setup.done") + root_name + "/models")\n    return config\n\n\ndef update_comfyui(config, ref):\n    """Move ComfyUI to another version and install what that version requires.\n\n    Custom nodes, settings, workflows and the database are not files of the ComfyUI\n    repository, so they stay as they are. Drive receives the environment only once the new\n    version is installed and consistent; an update that fails is undone."""\n    if (paths.PRIVATE / "processes.json").exists():\n        raise RuntimeError(t("update.running"))\n    if not re.fullmatch(r"[A-Za-z0-9._/-]+", ref) or ref.startswith("-"):\n        raise ValueError(t("setup.invalid_ref"))\n    drive.drive_root(config)  # The result is saved at the end: find out now if Drive is gone.\n    repo = paths.BASE / "environment/ComfyUI"\n    py = paths.BASE / ".venv/bin/python"\n    lock = "environment/requirements.lock.txt"\n    marker = paths.PRIVATE / "update-incomplete.json"\n\n    def git(*args):\n        return commands.run(["git", *args], app=True, capture=True, cwd=repo, timeout=600).strip()\n\n    def commit(name):\n        # The repository belongs to the application user: take nothing but a commit id from it.\n        try:\n            found = git("rev-parse", "--verify", "--quiet", name + "^{commit}")\n        except RuntimeError:\n            return ""\n        return found if re.fullmatch(r"[0-9a-f]{40,64}", found) else ""\n\n    def switch(target):\n        git("checkout", "--force", "--detach", target)\n        app_tree.app_action("prepare")  # The checkout puts back the models folder ComfyUI ships.\n\n    def package_names(listing):\n        return {re.split(r"==| @ ", line)[0] for line in listing.splitlines() if dependencies.LOCK_LINE.fullmatch(line)}\n\n    def undo(state):\n        """Back to the version and the packages recorded before the update began."""\n        packages = dependencies.checked_lock(app_tree.app_action("get", lock).decode())\n        index = dependencies.checked_torch_index(json.loads(app_tree.app_action("get", "environment/runtime.json")).get("torch_index_url"))\n        switch(state["commit"])\n        dependencies.install_packages("--index-strategy", "unsafe-best-match", "--extra-index-url", index, "-r", paths.BASE / lock)\n        added = sorted(package_names(commands.run([py, "-m", "pip", "freeze"], app=True, capture=True, timeout=300))\n                       - package_names(packages))\n        if added:\n            commands.run([py, "-m", "uv", "pip", "uninstall", "--python", py, *added], app=True)\n        marker.unlink()\n\n    if marker.exists():\n        # An earlier update failed and could not be undone then.\n        print(t("update.resuming"))\n        undo(json.loads(marker.read_text()))\n    git("fetch", "--tags", "--force", "origin")\n    state = {"commit": commit("HEAD"), "version": git("describe", "--tags", "--always")[:80]}\n    target = next(filter(None, map(commit, ("refs/tags/" + ref, "refs/remotes/origin/" + ref, ref))), "")\n    if not state["commit"] or not target:\n        raise ValueError(t("update.unknown_ref", ref=ref))\n    if target == state["commit"]:\n        print(t("update.already", version=state["version"]))\n        return False\n    print(t("update.starting", previous=state["version"], ref=ref))\n    dependencies.write_lock()\n    filesystem.private_write(marker, json.dumps(state))\n    try:\n        switch(target)\n        dependencies.install_packages(*(argument for name in ("requirements.txt", "manager_requirements.txt")\n                                        if (repo / name).exists() for argument in ("-r", repo / name)))\n        commands.run([py, "-m", "pip", "check"], app=True)\n    except Exception:\n        try:\n            undo(state)\n        except Exception:\n            print(t("update.undo_failed"))\n            raise\n        print(t("update.undone", version=state["version"]))\n        raise\n    marker.unlink()\n    version = git("describe", "--tags", "--always")[:80]\n    environment.checkpoint(config, force=True)\n    print(t("update.done", previous=state["version"], version=version))\n    return True\n', 'messages.py': '"""The texts the runtime shows, in the language of the notebook (translations.json)."""\n\nimport json\nfrom pathlib import Path\n\n\nMESSAGES = {\n    \'common.none\': \'无\',\n    \'common.unknown\': \'未知\',\n    \'common.present\': \'存在\',\n    \'common.yes\': \'是\',\n    \'common.no\': \'否\',\n    \'path.use_slashes\': \'相对路径请使用 /，不要包含盘符。\',\n    \'path.unsafe\': \'请指定安全的相对路径。\',\n    \'path.outside_folder\': \'目标路径超出了指定目录。\',\n    \'path.symlink\': \'目标路径包含符号链接。\',\n    \'command.running\': \'正在执行：\',\n    \'command.failed\': \'{stage} 失败：{failure}\',\n    \'command.exit_code\': \'{stage} 执行失败，退出码 {code}。请检查上方输出。\',\n    \'command.done\': \'完成：{stage}（{seconds:.0f} 秒）\',\n    \'command.timed_out\': \'未能在规定时间内完成\',\n    \'command.output_limit\': \'输出内容超过允许的大小\',\n    \'user.processes_remain\': \'无法停止 ComfyUI 用户的进程。请删除运行时并重新连接。\',\n    \'user.setpriv_missing\': \'缺少 {setpriv}。无法以受限权限运行 ComfyUI，安装已停止。\',\n    \'app.no_outputs\': \'没有可供打包的生成文件。\',\n    \'app.failed\': \'{action} 失败：{failure}\',\n    \'app.failed_detail\': \'{action} 失败：\',\n    \'venv.pip_fallback\': \'Notebook 的 pip 无法安装到虚拟环境，改用系统的 ensurepip。\',\n    \'venv.ensurepip_missing\': \'Python {version} 缺少 ensurepip，正在安装 {package}。\',\n    \'venv.package_failed\': \'无法安装 {package}。请在 Colab 中运行 apt-cache policy {package} 并检查可用版本及 APT 错误输出。\',\n    \'venv.ensurepip_unavailable\': \'安装 {package} 后，{python} 仍无法使用 ensurepip。请检查 Python 安装及 APT 软件包。\',\n    \'drive.needs_root\': \'保护 Google Drive 需要具有 root 权限的 Colab Linux 运行时。\',\n    \'drive.private_symlink\': \'受保护目录包含符号链接。\',\n    \'drive.private_owner\': \'受保护目录的所有者不是 root。\',\n    \'drive.mounted_elsewhere\': \'Google Drive 已挂载到其他位置。请删除运行时，并仅通过本 Notebook 挂载。\',\n    \'drive.no_mydrive\': \'在 Google Drive 中找不到 MyDrive。\',\n    \'drive.not_mounted\': \'Google Drive 尚未挂载，请先运行第 3 步初始化。\',\n    \'models.synced\': \'已从 Google Drive 复制模型：{copied} 个文件\',\n    \'models.linked\': \'直接从 Google Drive 读取模型（新增：{linked} 个）。首次使用模型时需要读取文件，因此首次加载较慢。\',\n    \'models.view_place\': \'无法将 {path} 用作模型读取目录。\',\n    \'models.view_failed\': \'无法准备从 Google Drive 直接读取模型（{reason}）。请将第 1 步的 MODEL_ACCESS 设为 copy，再运行第 3 步，将模型复制到 Colab 本地。\',\n    \'environment.archive_special_files\': \'环境归档中包含链接或特殊文件。\',\n    \'environment.archive_needs_data_filter\': \'安全恢复归档需要 Python 3.12 或更高版本（或支持 data_filter 的 Python 版本）。\',\n    \'environment.busy\': \'环境正在修改中，请等待安装完成后重新保存。\',\n    \'environment.unpinned_lock\': \'依赖列表包含未锁定版本的条目（可编辑安装、本地路径或 pip 选项）。请改为普通的 pip 软件包依赖。\',\n    \'cloudflared.download_failed\': \'下载 cloudflared 失败。\',\n    \'cloudflared.sha256_mismatch\': \'cloudflared 的 SHA256 与发布的校验值不一致，已终止操作。\',\n    \'setup.not_colab\': \'此安装过程仅适用于具有 root 权限的 Google Colab Linux GPU 运行时。\',\n    \'setup.already_done\': \'此运行时已完成初始化。如需重新初始化，请删除运行时并重新连接。\',\n    \'setup.invalid_ref\': \'ComfyUI 版本引用无效。\',\n    \'setup.interval_too_short\': \'自动保存间隔至少需要 60 秒。\',\n    \'setup.invalid_apt_package\': \'APT 软件包名称无效。\',\n    \'setup.no_gpu\': \'请在 Colab 的运行时设置中选择 GPU。\',\n    \'setup.fresh_environment\': \'不恢复已保存环境，正在创建全新环境。下次保存将在 Google Drive 中建立新版本。\',\n    \'setup.invalid_archive_name\': \'保存记录中的归档文件名无效。\',\n    \'setup.python_changed\': \'Python 版本与保存环境时不同，需要迁移环境；现有存档不会被覆盖。\',\n    \'setup.archive_sha256_mismatch\': \'已保存归档的 SHA256 不匹配，恢复已中止。\',\n    \'setup.restored\': \'已从 Google Drive 恢复环境。保存时间：\',\n    \'setup.restored_nodes\': \'已恢复的自定义节点：\',\n    \'setup.source_incomplete\': \'恢复的 ComfyUI 源码不完整。\',\n    \'setup.saved_apt_packages\': \'保存环境时使用的额外系统软件包：\',\n    \'setup.saved_apt_packages_hint\': \'如仍需要这些软件包，请将其添加到第 1 步的 APT_PACKAGES，并在新运行时重新执行；不会自动安装。\',\n    \'setup.no_torch\': \'未检测到标准 Colab PyTorch CUDA 环境，请检查 GPU 运行时。\',\n    \'setup.invalid_torch_mirror\': \'已保存的 PyTorch 镜像地址无效。\',\n    \'setup.copying_models\': \'正在将模型从 Google Drive 复制到 Colab（需要足够的本地存储空间）。\',\n    \'setup.done\': \'初始化完成。Google Drive 模型目录：MyDrive/\',\n    \'setup.invalid_model_access\': \'MODEL_ACCESS 必须是 direct 或 copy。\',\n    \'setup.sage_enabled\': \'正在使用 SageAttention。\',\n    \'setup.sage_unavailable\': \'此运行时无法使用 SageAttention，将使用标准注意力启动 ComfyUI。具体原因请查看上方输出。\',\n    \'download.hf_repository\': \'请将 Hugging Face 仓库 ID 填写为 作者/仓库。\',\n    \'download.hf_revision\': \'请指定版本引用。\',\n    \'download.civitai_version\': \'请填写正整数形式的 Civitai 模型版本 ID。\',\n    \'download.unknown_source\': \'请选择 Hugging Face 或 Civitai 作为下载来源。\',\n    \'download.hf_url\': \'Hugging Face 请提供 Files and versions 中具体文件的 URL（…/blob/… 或 …/resolve/…）。\',\n    \'download.civitai_url\': \'Civitai 请提供已选中版本的页面链接（…?modelVersionId=…）或直接下载链接。\',\n    \'download.unsupported_url\': \'请提供 https://huggingface.co/ 或 https://civitai.com/ 上的 URL。\',\n    \'download.filename_has_folder\': \'文件名中不要包含目录。请通过 MODEL_FOLDER 选择保存位置。\',\n    \'download.not_https\': \'下载地址必须是没有账号密码的 HTTPS URL。\',\n    \'download.local_redirect\': \'已拒绝重定向至本地服务。\',\n    \'download.unresolved_host\': \'无法解析下载地址的主机名。\',\n    \'download.private_redirect\': \'已拒绝重定向至私有 IP 地址。\',\n    \'download.redirect_without_target\': \'下载重定向未提供目标地址。\',\n    \'download.http_error\': \'模型下载失败（HTTP {status}）。请检查使用条款、访问权限和令牌。\',\n    \'download.not_a_model\': \'下载内容是 HTML/JSON，并非模型文件。请检查指定的文件。\',\n    \'download.empty\': \'下载到的文件为空。\',\n    \'download.too_many_redirects\': \'重定向次数过多。\',\n    \'download.host_not_allowed\': \'仅允许从 huggingface.co 或 civitai.com 下载。\',\n    \'download.url_has_query\': \'源 URL 不应包含查询参数或访问密钥。请通过输入表单生成 URL。\',\n    \'download.invalid_sha256\': \'SHA256 必须为 64 位十六进制字符。\',\n    \'download.exists\': \'Google Drive 中已经存在同名模型，请换一个文件名。\',\n    \'download.sha256_mismatch\': \'模型 SHA256 校验不通过，文件未保存。\',\n    \'download.no_filename\': \'网站未提供文件名，请手动填写。\',\n    \'download.exists_not_overwritten\': \'Google Drive 中已有同名模型，未覆盖现有文件。请换一个文件名。\',\n    \'download.saved\': \'已保存到 Google Drive，并复制到 Colab：\',\n    \'download.connection_failed\': \'模型下载连接失败，请检查网络连接和访问权限。\',\n    \'download.label\': \'正在下载\',\n    \'download.progress\': \'正在下载：{done:.0f} MB / {total:.0f} MB（{percent:.0f}%）\',\n    \'download.progress_size\': \'正在下载：{done:.0f} MB\',\n    \'download.saved_runtime\': \'已保存到当前运行时（未保存到 Google Drive）：\',\n    \'download.exists_runtime\': \'当前运行时已有同名模型，请换一个文件名。\',\n    \'proxy.unauthorized\': \'需要身份验证。请在浏览器中打开 Notebook 显示的 URL 并输入配对码。\',\n    \'proxy.cross_origin\': \'不允许来自其他来源的访问。\',\n    \'proxy.websocket_error\': \'WebSocket 转发错误：\',\n    \'proxy.upstream_down\': \'ComfyUI 正在启动或已经停止。\',\n    \'proxy.origin_rejected\': \'已拒绝其他站点的请求：Origin {origin} / Host {host}\',\n    \'proxy.access_required\': \'无法验证 Cloudflare Access 登录状态。请通过公开主机名 URL 连接。如果已登录仍出现此提示，请检查密钥 CF_ACCESS_AUD 中的 AUD 标签。\',\n    \'proxy.site_rejected\': \'已拒绝其他站点的请求：{method} {path}（Sec-Fetch-Site: {site}）\',\n    \'diagnosis.route\': \'  [{mark}] {status} {seconds:6.2f}秒 {size:>9}B {route}\',\n    \'diagnosis.assets\': \'  [{mark}] UI JS/CSS 共 {total} 个文件：失败 {failed} 个 / 未压缩 {megabytes:.1f}MB / 耗时 {elapsed:.2f}秒 / 压缩 {compressed}\',\n    \'diagnosis.failed_asset\': \'       {status} {seconds:.2f}秒 {route}\',\n    \'diagnosis.websocket\': \'  [{mark}] WebSocket 第一条消息：{kind}\',\n    \'diagnosis.not_running\': \'ComfyUI 尚未运行。请先运行 6-1、6-2 或 6-3，再执行诊断。\',\n    \'diagnosis.target_comfyui\': \'ComfyUI（127.0.0.1:8188）\',\n    \'diagnosis.target_proxy\': \'认证代理（127.0.0.1:8189）\',\n    \'diagnosis.target_public\': \'公共 URL（通过 Cloudflare）\',\n    \'diagnosis.failed_at\': \'诊断结果：问题发生在 {label}。之前的环节均正常。\',\n    \'diagnosis.all_ok\': \'诊断结果：Colab 侧直至公共 URL 的各环节均正常。如果界面无法显示，请检查电脑上的浏览器扩展、广告拦截器和安全软件。\',\n    \'diagnosis.colab_ok\': \'诊断结果：ComfyUI 和认证代理工作正常。这里无法检查 Colab 端口转发。如果界面无法显示，请确认浏览器登录的是运行此 Notebook 的 Google 账号，并检查第三方 Cookie 限制及浏览器扩展。\',\n    \'diagnosis.access_ok\': \'诊断结果：ComfyUI 工作正常。认证代理和 Cloudflare 侧需要 Access 登录，无法从这里检查。如果界面无法显示，请查看第 7 步的 tunnel.log，以及 Cloudflare 中的隧道和 Access 应用配置。\',\n    \'isolation.heading\': \'应用用户（{user}）可访问的本地服务：\',\n    \'isolation.comfyui\': \'ComfyUI\',\n    \'isolation.proxy\': \'认证代理\',\n    \'isolation.unix_sockets\': \'可成功连接的 Unix Socket（{listening} 个监听中有 {reachable} 个）：\',\n    \'isolation.kernel\': \'内核：\',\n    \'isolation.root_capabilities\': \'root 权限能力：\',\n    \'isolation.no_command\': \'找不到命令\',\n    \'isolation.available\': \'可用\',\n    \'isolation.unavailable\': \'不可用\',\n    \'supervisor.autosave_failed\': \'自动保存环境失败。结束使用前请手动执行保存单元格。\',\n    \'supervisor.save_before_restart_failed\': \'重新启动前保存失败\',\n    \'supervisor.restarting\': \'ComfyUI 已退出，将在 10 秒后重新启动。\',\n    \'supervisor.restart_limit\': \'ComfyUI 已达到最大重启次数，请检查 comfy.log。\',\n    \'start.already_running\': \'ComfyUI 已在运行。需要重启时，请先执行停止单元格。\',\n    \'start.proxy_exited_port\': \'认证代理已退出，8189 端口可能被占用。请查看代理日志 proxy.log。\',\n    \'start.proxy_not_started\': \'认证代理未能启动。\',\n    \'start.comfyui_timeout\': \'ComfyUI 在 5 分钟内未能启动，请检查日志单元格。\',\n    \'start.proxy_exited\': \'认证代理已退出，请查看 proxy.log。\',\n    \'start.url_auto_download\': \'使用远程连接时，请启用 Auto-download outputs，并设置电脑上的输出目录。\',\n    \'start.tunnel_exited\': \'Quick Tunnel 已退出，请查看 tunnel.log。\',\n    \'start.no_tunnel_url\': \'无法获取 Quick Tunnel URL。\',\n    \'start.tunnel_heading\': \'在 Comfy Desktop 的 Remote Connection 中输入以下 URL（也可使用普通浏览器打开）：\',\n    \'start.pairing_code\': \'配对码：{code}（仅限使用一次，有效期 {minutes} 分钟）\',\n    \'start.pairing_note\': \'打开 URL 后需输入配对码。要添加其他设备，请在第 6-2 步勾选 NEW_PAIRING_CODE 后重新运行。停止后 URL 及所有连接均失效。\',\n    \'start.colab_heading\': \'在当前浏览器中打开以下 URL（仅运行此 Notebook 的 Google 账号可访问）：\',\n    \'start.colab_note\': \'Comfy Desktop 和其他设备无法通过此方式连接。请在 ComfyUI 页面下载生成文件，或在第 9 步打包下载。\',\n    \'start.colab_unavailable\': \'无法获取 Colab 端口转发地址。请在 Colab Notebook 中执行。\',\n    \'start.keep_running\': \'使用 ComfyUI 时请让此单元格保持运行，以降低被 Colab 认定为空闲的概率。如需使用其他单元格，先点击 ■ 中断，结束后运行第 6-4 步恢复等待。\',\n    \'keep.not_running\': \'ComfyUI 尚未运行。请先使用 6-1、6-2 或 6-3 启动，再运行此单元格。\',\n    \'keep.status\': \'ComfyUI 运行中（已运行 {running}）。环境上次保存时间：{saved}。\',\n    \'keep.saved_minutes\': \'{minutes} 分钟前\',\n    \'keep.saved_none\': \'尚未保存（环境发生变化时会保存）\',\n    \'keep.saved_failed\': \'保存失败（请查看第 7 步）\',\n    \'keep.interrupted\': \'已停止等待，ComfyUI 仍在运行。完成其他单元格操作后，请执行第 6-4 步恢复等待。\',\n    \'keep.stopped\': \'ComfyUI 已停止，此单元格已结束。\',\n    \'stop.saved\': \'环境已保存到 Google Drive。\',\n    \'stop.done\': \'已停止。删除运行时之前请确认生成文件已保存到电脑。\',\n    \'update.running\': \'ComfyUI 正在运行。请先通过第 10 步停止（关闭 FLUSH_AND_UNMOUNT）。\',\n    \'update.unknown_ref\': \'ComfyUI 中不存在名为 {ref} 的版本。\',\n    \'update.already\': \'ComfyUI 已是 {version} 版本。\',\n    \'update.starting\': \'正在将 ComfyUI 从 {previous} 更新到 {ref}。\',\n    \'update.undone\': \'更新失败，已将 ComfyUI 恢复到 {version}。Google Drive 中的内容未改变。\',\n    \'update.undo_failed\': \'更新失败，而且无法恢复旧版本。请重新运行第 11 步。\',\n    \'update.incomplete\': \'上次 ComfyUI 更新未完成。请重新运行第 11 步，或删除运行时（Google Drive 中仍保留更新前的环境）。\',\n    \'update.resuming\': \'正在撤销上次未完成的更新。\',\n    \'update.done\': \'已将 ComfyUI 从 {previous} 更新到 {version} 并保存到 Google Drive。请执行 6-1、6-2 或 6-3 启动。\',\n    \'pairing.title\': \'连接到 ComfyUI\',\n    \'pairing.prompt\': \'请输入 Notebook 显示的配对码。\',\n    \'pairing.button\': \'连接\',\n    \'pairing.wrong\': \'配对码不正确，剩余尝试次数：{remaining}。\',\n    \'pairing.closed\': \'此配对码不可使用。请在 Notebook 的第 6-2 步勾选 NEW_PAIRING_CODE 重新生成。\',\n    \'pairing.not_running\': \'请先通过第 6-2 步启动。\',\n    \'access.invalid_hostname\': \'公开主机名无效，请按照 comfy.example.com 的格式填写。\',\n    \'access.invalid_audience\': \'AUD 标签无效。请将 Access 应用的 64 字符 Application Audience (AUD) Tag 保存为 Colab 密钥 CF_ACCESS_AUD。\',\n    \'access.invalid_token\': \'隧道令牌无效。请将以 eyJ 开头的原始令牌保存为 Colab 密钥 CF_TUNNEL_TOKEN。\',\n    \'access.tunnel_exited\': \'命名隧道已退出。请检查令牌和第 7 步的 tunnel.log。\',\n    \'access.unprotected\': \'未检测到 {hostname} 将访客重定向到 Cloudflare Access 登录页面（状态：{status}）。请检查该主机名对应的 Access 应用和 DNS 记录。刚修改配置时请等待几分钟后重试。程序尚未启动。\',\n    \'access.heading\': \'在 Comfy Desktop 的 Remote Connection 中输入以下 URL（普通浏览器也可打开）：\',\n    \'access.note\': \'打开 URL 后将显示 Cloudflare Access 登录界面，只有获授权账号登录的设备可以连接。停止后此 URL 将无法访问。\',\n    \'access.wrong_target\': \'隧道的公开主机名并非全部指向 http://127.0.0.1:8189（当前设置：{services}）。请在 Cloudflare 隧道配置中将服务类型设为 HTTP、地址设为 127.0.0.1:8189，再重新运行。尚未连接。\',\n    \'access.target_unknown\': \'提示：无法读取隧道的转发目标。请在 Cloudflare 的隧道配置中确认公开主机名指向 http://127.0.0.1:8189。\',\n    \'menu.stop\': \'停止\',\n    \'menu.title\': \'停止 ComfyUI\',\n    \'menu.explain\': \'将环境保存到 Google Drive 后断开连接。生成文件不会保存到 Google Drive；删除运行时会丢失尚未下载到电脑的输出。\',\n    \'menu.delete\': \'断开连接并删除运行时\',\n    \'menu.disconnect\': \'断开连接\',\n    \'menu.cancel\': \'取消\',\n    \'menu.working\': \'正在保存并停止。完成后此页面会断开连接。\',\n    \'menu.stopped\': \'已断开连接。若要重新启动，请运行 Notebook 中的 6-1、6-2 或 6-3。\',\n    \'menu.deleted\': \'已断开连接，删除运行时还需要一些时间。请在 Colab 确认运行时已断开。\',\n    \'menu.failed\': \'无法停止。请使用 Notebook 的第 10 步停止。\',\n    \'menu.slow\': \'操作耗时较长。如果一直没有进展，请使用 Notebook 的第 10 步停止，或在 Colab 中删除运行时。\',\n    \'menu.repository\': \'打开 GitHub 仓库\',\n}\nif MESSAGES is None:\n    # Run from the source tree: the Japanese messages of translations.json next to this package.\n    MESSAGES = {f"{section}.{name}": entry["JP"] for section, entries in json.loads(\n        Path(__file__).resolve().parent.parent.joinpath("translations.json").read_text(encoding="utf-8"))["runtime"].items()\n        for name, entry in entries.items()}\n\n\ndef t(key, **values):\n    """The message translations.json holds as runtime.<key>, with its values filled in."""\n    return MESSAGES[key].format(**values)\n', 'pairing.py': '"""Pairing codes: how a browser is let in through the public URL (6-2)."""\n\nimport hmac\nimport json\nimport re\nimport secrets\nimport time\n\nfrom . import filesystem, paths\nfrom .messages import t\n\n\nPAIRING_PATH = "/_colab/pair"\nPAIRING_SECONDS = 600\nPAIRING_ATTEMPTS = 5\n\n\nPAIRING_PAGE = """<!doctype html>\n<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n<title>{title}</title>\n<style>\nbody {{ font-family: system-ui, sans-serif; background: #202020; color: #eee; margin: 0;\n       min-height: 100vh; display: flex; align-items: center; justify-content: center; }}\nmain {{ width: 20rem; padding: 1.5rem; }}\nh1 {{ font-size: 1.3rem; }}\ninput, button {{ font: inherit; width: 100%; box-sizing: border-box; padding: .6rem; border-radius: .4rem; }}\ninput {{ font-size: 1.4rem; letter-spacing: .2em; text-align: center; background: #111; color: #eee; border: 1px solid #666; }}\nbutton {{ margin-top: .8rem; background: #3b82f6; color: #fff; border: 0; }}\n.notice {{ color: #fca5a5; }}\n</style></head>\n<body><main><h1>{title}</h1><p>{prompt}</p><p class="notice">{notice}</p>\n<form method="post" action="{path}">\n<input name="code" inputmode="numeric" autocomplete="one-time-code" maxlength="16" autofocus required>\n<button type="submit">{button}</button>\n</form></main></body></html>\n"""\n\n\ndef new_pairing_code():\n    """Issue the code that lets one browser in. It replaces any earlier one."""\n    code = "".join(secrets.choice("0123456789") for _ in range(8))\n    filesystem.private_write(paths.PRIVATE / "pairing.json", json.dumps(\n        {"code": code, "expires": time.time() + PAIRING_SECONDS, "attempts": PAIRING_ATTEMPTS}))\n    return code\n\n\ndef redeem_pairing_code(supplied):\n    """(accepted, tries left). A code is good for one browser, a few minutes and a few wrong\n    guesses; after any of these there is no code until the notebook issues another."""\n    path = paths.PRIVATE / "pairing.json"\n    try:\n        state = json.loads(path.read_text())\n    except (OSError, ValueError):\n        return False, 0\n    if time.time() > state["expires"]:\n        path.unlink(missing_ok=True)\n        return False, 0\n    if hmac.compare_digest(re.sub(r"[\\s-]", "", supplied).encode(), state["code"].encode()):\n        path.unlink(missing_ok=True)\n        return True, 0\n    state["attempts"] -= 1\n    if state["attempts"] <= 0:\n        path.unlink(missing_ok=True)\n        return False, 0\n    filesystem.private_write(path, json.dumps(state))\n    return False, state["attempts"]\n\n\ndef show_pairing(config):\n    code = new_pairing_code()\n    print(t("start.tunnel_heading"))\n    print(config["remote_url"] + "/")\n    print(t("start.pairing_code", code=code[:4] + "-" + code[4:], minutes=PAIRING_SECONDS // 60))\n    print(t("start.pairing_note"))\n\n\ndef issue_pairing_code(config):\n    """Let one more browser in while ComfyUI runs behind the tunnel."""\n    if config.get("mode") != "tunnel" or not config.get("remote_url"):\n        raise RuntimeError(t("pairing.not_running"))\n    show_pairing(config)\n', 'paths.py': '"""Where the runtime keeps things on the Colab machine.\n\nOther modules read these as paths.NAME: the app action and the tests point them elsewhere."""\n\nfrom pathlib import Path\n\n\nBASE = Path("/content/comfy-session")  # The application user\'s tree: ComfyUI, its environment and models.\nPRIVATE = Path("/content/.comfy-private")  # Root only: Drive, keys, logs and the session settings.\nDRIVE_MOUNT = PRIVATE / "drive"\nCONFIG = PRIVATE / "session.json"\nDRIVE_MODELS = Path("/content/comfy-drive-models")  # The read-only view of Drive\'s models folder.\nMODEL_FOLDERS = ("checkpoints", "diffusion_models", "unet", "text_encoders", "clip", "clip_vision", "vae",\n                 "loras", "controlnet", "upscale_models", "embeddings", "audio_encoders", "model_patches")\n', 'proxy.py': '"""The proxy in front of ComfyUI: the only way in from outside the machine."""\n\nimport asyncio\nimport hmac\nimport json\nimport re\nimport time\nfrom urllib.parse import urlsplit, parse_qs\n\nfrom . import cloudflare, pairing, stop_button\nfrom .messages import t\n\n\nCOOKIE = "__Host-comfy_colab"\n\n\nCOOKIE_SECONDS = 24 * 3600\nACCESS_KEYS_REFRESH = 30\n\n\ndef create_proxy(config, upstream="http://127.0.0.1:8188", secure_cookie=True):\n    """Streaming proxy in front of ComfyUI.\n\n    Behind the tunnel it lets in only browsers that hold the session cookie, which a pairing\n    code buys once; the cookie also rides Desktop\'s download session. Behind Colab\'s port\n    forwarding Google has already let in the runtime\'s owner only, so it just keeps other\n    sites from sending requests through that browser. Behind Cloudflare Access it lets in\n    only requests that carry Access\'s signature for this application."""\n    from aiohttp import web, ClientError, ClientSession, ClientTimeout, WSMsgType, DummyCookieJar\n    import html\n    forwarded = config.get("mode") == "colab"\n    access = config.get("access") if config.get("mode") == "access" else None\n    if access:\n        import jwt\n    signing_keys = {}\n    keys_fetched = [float("-inf")]\n    verified = {}  # token -> expiry: one page load sends the same token a few hundred times.\n    cookie_secret = config.get("cookie_secret", "")\n    cookie_name = COOKIE if secure_cookie else "comfy_colab"\n    origins = set(config.get("origins", ()))\n    hop = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization",\n           "te", "trailer", "transfer-encoding", "upgrade", "content-length"}\n    client_key = web.AppKey("client", ClientSession)\n    # Vite names frontend files after their content, so they can be cached for good.\n    hashed_asset = re.compile(r"/assets/[\\w.-]+-[A-Za-z0-9_-]{8}\\.(?:js|css|json|svg|ttf|woff2?)")\n    compressible = ("text/", "application/javascript", "application/json", "image/svg+xml")\n\n    def matches(supplied, secret):\n        # Bytes: comparing str raises on the non-ASCII input anyone can send.\n        return bool(secret) and hmac.compare_digest(supplied.encode(), secret.encode())\n\n    async def lifecycle(app):\n        app[client_key] = ClientSession(timeout=ClientTimeout(total=None, sock_connect=30),\n                                      auto_decompress=False, cookie_jar=DummyCookieJar())\n        yield\n        await app[client_key].close()\n\n    async def signing_key(kid):\n        if kid not in signing_keys and time.monotonic() - keys_fetched[0] >= ACCESS_KEYS_REFRESH:\n            # Cloudflare rotates its keys, so an unknown one is looked up, but not for every bad request.\n            keys_fetched[0] = time.monotonic()\n            async with ClientSession(timeout=ClientTimeout(total=15)) as session:\n                async with session.get(cloudflare.access_certs_url(access["team"])) as res:\n                    published = (await res.json(content_type=None))["keys"]\n            signing_keys.update({key["kid"]: jwt.PyJWK.from_dict(key).key for key in published})\n        return signing_keys[kid]\n\n    async def signed_by_access(request):\n        """Cloudflare adds this token to what it lets through. Checking it here means a request\n        that reaches the tunnel some other way, or for another application, is still refused."""\n        token = request.headers.get("Cf-Access-Jwt-Assertion", "")\n        if not token or len(token) > 8192:\n            return False\n        if verified.get(token, 0) > time.time():\n            return True\n        try:\n            key = await signing_key(jwt.get_unverified_header(token)["kid"])\n            claims = jwt.decode(token, key, algorithms=["RS256"], audience=access["aud"],\n                                issuer="https://" + access["team"], options={"require": ["exp", "iat", "aud", "iss"]})\n        except (jwt.PyJWTError, ClientError, KeyError, TypeError, ValueError, OSError, asyncio.TimeoutError):\n            return False\n        if len(verified) >= 256:\n            verified.clear()\n        verified[token] = claims["exp"]\n        return True\n\n    noted = [0.0, 0]  # When the current minute of the log began, and the lines written in it.\n\n    def note(message):\n        """A line in proxy.log, but no more of them than a visitor\'s requests may fill it with."""\n        if time.monotonic() - noted[0] >= 60:\n            noted[:] = [time.monotonic(), 0]\n        noted[1] += 1\n        if noted[1] <= 30:\n            print(message, flush=True)\n\n    def refuse_other_sites(request, quiet=False):\n        """Stop what another site sends through a signed-in browser. quiet: the sender has not\n        signed in, so it could be anyone, and gets no line in the log."""\n        origin = request.headers.get("Origin")\n        if origin and origin not in {"https://" + request.host, "http://" + request.host} | origins:\n            if not quiet:\n                note(t("proxy.origin_rejected", origin=origin[:200], host=request.host[:200]))\n            raise web.HTTPForbidden(text=t("proxy.cross_origin"))\n        # A link, an image or a frame on another site carries no Origin, but the browser says where\n        # it comes from. From elsewhere only opening the page itself is taken, not ComfyUI\'s addresses:\n        # ComfyUI refuses those itself only in its recent versions.\n        site = request.headers.get("Sec-Fetch-Site", "").lower()\n        if site in ("cross-site", "same-site"):\n            opening = (request.method == "GET" and request.path in ("/", "/index.html")\n                       and request.headers.get("Sec-Fetch-Mode", "").lower() == "navigate"\n                       and request.headers.get("Sec-Fetch-Dest", "").lower() == "document")\n            if not opening:\n                if not quiet:\n                    note(t("proxy.site_rejected", method=request.method[:16], path=request.path[:200], site=site))\n                raise web.HTTPForbidden(text=t("proxy.cross_origin"))\n        return origin\n\n    def passed_on(res, keep_length):\n        """ComfyUI\'s response headers for the browser. A custom node must not replace the cookie\n        that signs the browser in, and no other site may show these pages in a frame."""\n        dropped = hop - {"content-length"} if keep_length else hop\n        headers = [(k, v) for k, v in res.headers.items() if k.lower() not in dropped and not (\n            k.lower() == "set-cookie" and (v.split("=", 1)[0].strip() in {COOKIE, "comfy_colab"}\n                                           or v.lstrip().startswith("CF_")))]\n        headers.append(("Content-Security-Policy", "frame-ancestors \'self\'"))\n        if "X-Frame-Options" not in res.headers:\n            headers.append(("X-Frame-Options", "SAMEORIGIN"))\n        return headers\n\n    def pairing_page(notice="", status=200):\n        page = pairing.PAIRING_PAGE.format(path=pairing.PAIRING_PATH, notice=html.escape(notice), title=html.escape(t("pairing.title")),\n                                           prompt=html.escape(t("pairing.prompt")), button=html.escape(t("pairing.button")))\n        # Not no-referrer: a browser then submits the form with "Origin: null", which is refused below.\n        return web.Response(text=page, status=status, content_type="text/html", headers={\n            "Cache-Control": "no-store", "Referrer-Policy": "same-origin", "X-Frame-Options": "DENY",\n            "Content-Security-Policy": "default-src \'none\'; style-src \'unsafe-inline\'; form-action \'self\'; "\n                                       "frame-ancestors \'none\'; base-uri \'none\'"})\n\n    async def pair(request):\n        """The one page anyone can reach: it trades the code shown in the notebook for the cookie."""\n        if request.method != "POST":\n            return pairing_page()\n        refuse_other_sites(request, quiet=True)\n        length = request.content_length\n        if length is None or length > 256:  # Nothing unauthenticated gets to send more.\n            raise web.HTTPBadRequest()\n        try:\n            form = parse_qs((await request.content.readexactly(length)).decode(errors="replace"))\n        except asyncio.IncompleteReadError:\n            raise web.HTTPBadRequest() from None\n        accepted, remaining = pairing.redeem_pairing_code(form.get("code", [""])[0])\n        if not accepted:\n            return pairing_page(t("pairing.wrong", remaining=remaining) if remaining else t("pairing.closed"), 401)\n        response = web.HTTPSeeOther(location="/")\n        response.set_cookie(cookie_name, cookie_secret, max_age=COOKIE_SECONDS,\n                            secure=secure_cookie, httponly=True, samesite="Lax", path="/")\n        response.headers["Cache-Control"] = "no-store"\n        response.headers["Referrer-Policy"] = "no-referrer"\n        raise response\n\n    async def stop_endpoint(request):\n        """The stop button\'s request, and afterwards its questions about how that is going."""\n        if request.method == "POST":\n            length = request.content_length\n            if length is None or length > 256:\n                raise web.HTTPBadRequest()\n            try:\n                delete_runtime = json.loads(await request.content.readexactly(length))["delete_runtime"]\n            except (asyncio.IncompleteReadError, ValueError, KeyError, TypeError):\n                raise web.HTTPBadRequest() from None\n            if not isinstance(delete_runtime, bool):\n                raise web.HTTPBadRequest()\n            stop_button.request_stop(delete_runtime)\n        return web.json_response({"state": stop_button.stop_request_state()}, status=202 if request.method == "POST" else 200,\n                                 headers={"Cache-Control": "no-store"})\n\n    async def handler(request):\n        if access:\n            if not await signed_by_access(request):\n                raise web.HTTPUnauthorized(text=t("proxy.access_required"))\n        elif not forwarded and not matches(request.cookies.get(cookie_name, ""), cookie_secret):\n            if request.path == pairing.PAIRING_PATH:\n                return await pair(request)\n            if request.method == "GET" and "text/html" in request.headers.get("Accept", ""):\n                # A browser opening the address: take it to where the code goes.\n                raise web.HTTPSeeOther(location=pairing.PAIRING_PATH, headers={"Cache-Control": "no-store"})\n            raise web.HTTPUnauthorized(text=t("proxy.unauthorized"))\n        if request.path == pairing.PAIRING_PATH:\n            raise web.HTTPSeeOther(location="/")\n        origin = refuse_other_sites(request)\n        if request.path == stop_button.STOP_SCRIPT_PATH:\n            return web.Response(text=stop_button.stop_button_script(), content_type="application/javascript", charset="utf-8",\n                                headers={"Cache-Control": "no-store"})\n        if request.path == stop_button.STOP_PATH:\n            return await stop_endpoint(request)\n        # What proves the visitor\'s identity stops here: custom nodes get neither the cookie nor Access\'s token.\n        headers = {k: v for k, v in request.headers.items()\n                   if k.lower() not in hop | {"cookie", "authorization"} and not k.lower().startswith("cf-access-")}\n        if origin:\n            # ComfyUI refuses a request whose Host and Origin differ, and a forwarder may rewrite Host.\n            headers = {k: v for k, v in headers.items() if k.lower() != "host"}\n            headers["Host"] = urlsplit(origin).netloc\n        # ComfyUI answers 403 to cross-site requests, which includes following the\n        # connection URL from a link (e.g. Colab output). Loading the root page is safe.\n        if request.method == "GET" and request.path == "/":\n            headers = {k: v for k, v in headers.items() if k.lower() != "sec-fetch-site"}\n        page = request.method == "GET" and request.path in ("/", "/index.html")\n        if page:\n            # Asked for uncompressed: the stop button goes into this page before it is passed on.\n            headers = {k: v for k, v in headers.items() if k.lower() != "accept-encoding"}\n            headers["Accept-Encoding"] = "identity"\n        # Preserve other backend cookies, but never disclose the proxy credential to custom nodes.\n        # Behind Colab\'s forwarding its own sign-in cookies cannot be told apart, so none are passed on.\n        backend_cookies = {} if forwarded else {\n            k: v for k, v in request.cookies.items() if k not in {COOKIE, "comfy_colab"} and not k.startswith("CF_")}\n        if backend_cookies:\n            from http.cookies import SimpleCookie\n            cookies = SimpleCookie()\n            for k, v in backend_cookies.items():\n                cookies[k] = v\n            headers["Cookie"] = cookies.output(header="", sep=";").strip()\n        url = upstream + request.rel_url.raw_path_qs\n        if request.headers.get("Upgrade", "").lower() == "websocket":\n            # Each hop negotiates its own handshake. Forwarding the browser\'s\n            # permessage-deflate offer makes ComfyUI compress frames this client cannot read.\n            ws_headers = {k: v for k, v in headers.items() if not k.lower().startswith("sec-websocket-")}\n            protocols = [p.strip() for p in request.headers.get("Sec-WebSocket-Protocol", "").split(",") if p.strip()]\n            async with request.app[client_key].ws_connect(url, headers=ws_headers, protocols=protocols,\n                                                        max_msg_size=0) as remote:\n                local = web.WebSocketResponse(max_msg_size=0, protocols=[remote.protocol] if remote.protocol else ())\n                await local.prepare(request)\n\n                async def relay(source, dest):\n                    async for msg in source:\n                        if msg.type == WSMsgType.TEXT:\n                            await dest.send_str(msg.data)\n                        elif msg.type == WSMsgType.BINARY:\n                            await dest.send_bytes(msg.data)\n                        elif msg.type in {WSMsgType.CLOSE, WSMsgType.CLOSED, WSMsgType.ERROR}:\n                            if msg.type == WSMsgType.ERROR:\n                                note(t("proxy.websocket_error") + " " + repr(source.exception()))\n                            break\n                tasks = [asyncio.create_task(relay(local, remote)), asyncio.create_task(relay(remote, local))]\n                try:\n                    await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)\n                finally:\n                    for task in tasks:\n                        task.cancel()\n                    await asyncio.gather(*tasks, return_exceptions=True)\n                    await local.close()\n                return local\n        try:\n            async with request.app[client_key].request(request.method, url, headers=headers,\n                    data=request.content.iter_chunked(1024 * 1024), allow_redirects=False) as res:\n                page = (page and res.status == 200 and "Content-Encoding" not in res.headers\n                        and res.headers.get("Content-Type", "").lower().startswith("text/html"))\n                if page:\n                    response = web.Response(body=stop_button.with_stop_button(await res.read()), headers=passed_on(res, False))\n                else:\n                    response = web.StreamResponse(status=res.status, headers=passed_on(res, True))\n                response.headers["Referrer-Policy"] = "no-referrer"\n                complete = request.method == "GET" and res.status == 200\n                # ComfyUI marks the pictures it ships (template thumbnails, icons) as good for a day.\n                # The browser may keep those, so they are not fetched again on every look; nothing in\n                # between may. Generated files come from /api and /view addresses and are never kept.\n                kept = re.search(r"max-age=\\d+", res.headers.get("Cache-Control", ""))\n                if complete and hashed_asset.fullmatch(request.path):\n                    response.headers["Cache-Control"] = "private, max-age=31536000, immutable"\n                elif complete and kept and "." in request.path.rsplit("/", 1)[-1] and not request.path.startswith("/api/"):\n                    response.headers["Cache-Control"] = "private, " + kept.group(0)\n                else:\n                    response.headers["Cache-Control"] = "no-store"\n                # ComfyUI sends its 26 MB frontend uncompressed and the tunnel is the slow hop.\n                if (complete and "Content-Encoding" not in res.headers\n                        and res.headers.get("Content-Type", "").lower().startswith(compressible)\n                        and "gzip" in request.headers.get("Accept-Encoding", "").lower()):\n                    response.headers["Vary"] = "Accept-Encoding"\n                    response.enable_compression(web.ContentCoding.gzip)\n                if page:\n                    return response\n                await response.prepare(request)\n                async for chunk in res.content.iter_chunked(1024 * 1024):\n                    await response.write(chunk)\n                await response.write_eof()\n                return response\n        except (ConnectionError, OSError):\n            raise web.HTTPBadGateway(text=t("proxy.upstream_down")) from None\n\n    app = web.Application(client_max_size=1024 ** 3)\n    app.cleanup_ctx.append(lifecycle)\n    app.router.add_route("*", "/{path:.*}", handler)\n    return app\n', 'service.py': '"""Starting and stopping ComfyUI with its proxy and tunnel (cells 6 and 10)."""\n\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport signal\nimport subprocess\nimport threading\nimport time\nfrom urllib.parse import urlsplit\n\nfrom . import app_tree, cloudflare, commands, environment, filesystem, pairing, paths, stop_button\nfrom .messages import t\n\n\n# What a session adds to the saved settings, and stopping takes away again.\nSESSION_KEYS = ("mode", "cookie_secret", "remote_url", "origins", "access")\n\n\ndef load_config():\n    """The settings setup saved for this runtime, with those of the running session if any."""\n    return json.loads(paths.CONFIG.read_text())\n\n\ndef run_diagnosis():\n    """diagnose() as a process of its own (cell 7), with its output shown in the notebook."""\n    commands.run(commands.own_command("diagnose"), cwd=commands.OWN_DIRECTORY, timeout=600)\n\n\ndef backend_command(config):\n    repo = paths.BASE / "environment/ComfyUI"\n    return [str(paths.BASE / ".venv/bin/python"), str(repo / "main.py"),\n            "--listen", "127.0.0.1", "--port", "8188", "--enable-manager",\n            "--base-directory", str(repo), "--user-directory", str(repo / "user"),\n            "--input-directory", str(paths.BASE / "input"), "--output-directory", str(paths.BASE / "output"),\n            "--temp-directory", str(paths.BASE / "temp"), "--extra-model-paths-config", str(paths.BASE / "models.yaml"),\n            "--database-url", "sqlite:///" + str(paths.BASE / "database/comfyui.db"),\n            # Only where setup installed and tried it: a runtime set up before that has no package.\n            *(["--use-sage-attention"] if config.get("sage_attention") else []),\n            # One named optimization: --fast alone turns on every experimental one ComfyUI has.\n            *(["--fast", "fp16_accumulation"] if config.get("fp16_accumulation") else [])]\n\n\ndef supervise(config):\n    # Background process persists independently of the notebook cell.\n    def save_loop():\n        while True:\n            time.sleep(config["checkpoint_seconds"])\n            try:\n                environment.checkpoint(config)\n            except Exception as exc:\n                filesystem.private_write(paths.PRIVATE / "checkpoint-status.json", json.dumps({"ok": False, "error": str(exc)}))\n                print(t("supervisor.autosave_failed"), flush=True)\n    threading.Thread(target=save_loop, daemon=True).start()\n    env = commands.application_environment()\n    for _ in range(5):\n        with (paths.PRIVATE / "comfy.log").open("a") as log:\n            child = subprocess.Popen(commands.app_command(backend_command(config)), cwd=paths.BASE / "environment/ComfyUI", env=env,\n                                     stdout=log, stderr=subprocess.STDOUT)\n            child.wait()\n        try:\n            environment.checkpoint(config, force=True)\n        except Exception:\n            print(t("supervisor.save_before_restart_failed"), flush=True)\n        print(t("supervisor.restarting"), flush=True)\n        time.sleep(10)\n    print(t("supervisor.restart_limit"), flush=True)\n\n\ndef redact(text):\n    """Hide credentials and the tunnel address in text that is about to be displayed."""\n    text = re.sub(r"(?i)(access_key=|Bearer\\s+)[A-Za-z0-9_.-]+", r"\\1[REDACTED]", text)\n    text = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", text)\n    text = re.sub(r"eyJ[A-Za-z0-9_+/=-]{20,}(\\.[A-Za-z0-9_+/=-]+)*", "[REDACTED]", text)  # Tunnel and Access tokens.\n    return re.sub(r"[a-z0-9-]+\\.trycloudflare\\.com", "[REDACTED].trycloudflare.com", text)\n\n\ndef colab_forwarded_address(port):\n    """Where Colab serves a port of this runtime, to the browser signed in as its owner only."""\n    from google.colab import output\n    parts = urlsplit(str(output.eval_js(f"google.colab.kernel.proxyPort({port})")))\n    if parts.scheme != "https" or not parts.netloc:\n        raise RuntimeError(t("start.colab_unavailable"))\n    return f"https://{parts.netloc}/"\n\n\ndef start_with_access(config, hostname, audience, tunnel_token):\n    """Start behind a named Cloudflare tunnel whose hostname Cloudflare Access protects."""\n    access = cloudflare.access_settings(hostname, audience)\n    token = (tunnel_token or "").strip()\n    if not re.fullmatch(r"[A-Za-z0-9+/=_-]{40,4096}", token):\n        raise ValueError(t("access.invalid_token"))\n    if (paths.PRIVATE / "processes.json").exists():\n        raise RuntimeError(t("start.already_running"))\n    access["team"] = cloudflare.access_team_domain(access["hostname"])\n    start(config, "access", access=access, tunnel_token=token)\n\n\ndef start(config, mode, access=None, tunnel_token=""):\n    """Start ComfyUI and make it reachable: mode "colab" through Colab\'s port forwarding,\n    "tunnel" through a Quick Tunnel that asks each browser for a pairing code, "access"\n    through a named tunnel on a hostname that Cloudflare Access protects."""\n    import requests\n    if mode not in ("colab", "tunnel", "access") or (mode == "access") != bool(access and tunnel_token):\n        raise ValueError(mode)\n    if (paths.PRIVATE / "processes.json").exists():\n        raise RuntimeError(t("start.already_running"))\n    environment.refuse_unfinished_update()\n    for name in SESSION_KEYS:\n        config.pop(name, None)\n    for name in ("stop-request.json", "stop-status.json"):\n        (paths.PRIVATE / name).unlink(missing_ok=True)\n    config["mode"] = mode\n    if mode == "colab":\n        address = colab_forwarded_address(8189)\n        config["origins"] = [address.rstrip("/")]\n    elif mode == "tunnel":\n        # A cookie from an earlier start (browser, Desktop) stops working.\n        config["cookie_secret"] = secrets.token_urlsafe(32)\n    else:\n        config["access"] = access  # Not the tunnel\'s token: only cloudflared is given that.\n    filesystem.private_write(paths.CONFIG, json.dumps(config))\n    processes = {}\n\n    def launch(name):\n        with (paths.PRIVATE / (name + ".log")).open("a") as log:\n            process = subprocess.Popen(commands.own_command(name.replace("supervisor", "supervise")),\n                                       cwd=commands.OWN_DIRECTORY, stdout=log, stderr=subprocess.STDOUT,\n                                       start_new_session=True)\n        processes[name] = process.pid\n        filesystem.private_write(paths.PRIVATE / "processes.json", json.dumps(processes))\n        return process\n    try:\n        # A process left behind by a custom node could take the proxy\'s port and receive the tunnel.\n        commands.stop_app_processes()\n        proxy = launch("proxy")\n        for _ in range(30):\n            if proxy.poll() is not None:\n                raise RuntimeError(t("start.proxy_exited_port"))\n            try:\n                # Its answer before ComfyUI runs: no cookie behind the tunnel, no backend behind Colab.\n                if requests.get("http://127.0.0.1:8189/", timeout=3).status_code == (502 if mode == "colab" else 401):\n                    break\n            except requests.RequestException:\n                pass\n            time.sleep(1)\n        else:\n            raise RuntimeError(t("start.proxy_not_started"))\n        launch("supervisor")\n        deadline = time.monotonic() + 300\n        while time.monotonic() < deadline:\n            try:\n                if requests.get("http://127.0.0.1:8188/system_stats", timeout=3).ok:\n                    break\n            except requests.RequestException:\n                pass\n            time.sleep(2)\n        else:\n            raise RuntimeError(t("start.comfyui_timeout"))\n        if proxy.poll() is not None:\n            raise RuntimeError(t("start.proxy_exited"))\n        if mode == "colab":\n            print(t("start.colab_heading"))\n            print(address)\n            print(t("start.colab_note"))\n            print(t("start.keep_running"))\n            stop_button.follow_stop_requests(config)\n            return\n        logfile = paths.PRIVATE / "tunnel.log"\n        logfile.write_text("")\n        with logfile.open("a") as log:\n            if mode == "access":\n                # The token goes in the environment: a command line can be read by every user here.\n                tunnel = subprocess.Popen([str(paths.PRIVATE / "cloudflared"), "tunnel", "--no-autoupdate",\n                                           "--protocol", cloudflare.TUNNEL_PROTOCOL, "run"],\n                                          env={**os.environ, "TUNNEL_TOKEN": tunnel_token},\n                                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n            else:\n                tunnel = subprocess.Popen([str(paths.PRIVATE / "cloudflared"), "tunnel", "--no-autoupdate",\n                        "--protocol", cloudflare.TUNNEL_PROTOCOL, "--url", "http://127.0.0.1:8189"],\n                        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n        processes["tunnel"] = tunnel.pid\n        filesystem.private_write(paths.PRIVATE / "processes.json", json.dumps(processes))\n        if mode == "access":\n            services = connected = None\n            for second in range(60):\n                log = logfile.read_text()\n                services = cloudflare.tunnel_services(log)\n                if connected is None and "Registered tunnel connection" in log:\n                    connected = second\n                # Cloudflare sends the tunnel its settings right after it connects.\n                if services is not None or (connected is not None and second - connected >= 15):\n                    break\n                if tunnel.poll() is not None:\n                    raise RuntimeError(t("access.tunnel_exited"))\n                time.sleep(1)\n            if services is None:\n                print(t("access.target_unknown"))\n            elif not cloudflare.tunnel_leads_to_proxy(services):\n                raise RuntimeError(t("access.wrong_target", services=", ".join(services)[:300]))\n            print(t("access.heading"))\n            print("https://" + access["hostname"] + "/")\n            print(t("access.note"))\n            print(t("start.url_auto_download"))\n            print(t("start.keep_running"))\n            stop_button.follow_stop_requests(config)\n            return\n        for _ in range(90):\n            match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", logfile.read_text())\n            if match:\n                url = match.group(0)\n                config["remote_url"] = url\n                filesystem.private_write(paths.CONFIG, json.dumps(config))\n                pairing.show_pairing(config)\n                print(t("start.url_auto_download"))\n                print(t("start.keep_running"))\n                stop_button.follow_stop_requests(config)\n                return\n            if tunnel.poll() is not None:\n                raise RuntimeError(t("start.tunnel_exited"))\n            time.sleep(1)\n        raise RuntimeError(t("start.no_tunnel_url"))\n    except Exception:\n        stop(config, save=False)\n        raise\n\n\nKEEP_RUNNING_SECONDS = 5  # How often cell 6-4 looks whether ComfyUI still runs.\nKEEP_RUNNING_REFRESH = 60  # How often it rewrites its status line.\n\n\ndef keep_running(show=None):\n    """Cell 6-4: stay busy while ComfyUI runs, so that Colab does not take the notebook for idle.\n\n    Colab disconnects a runtime whose notebook does nothing; work on ComfyUI\'s page is not seen\n    there, but a cell that is running is. The wait ends when ComfyUI is stopped (cell 10 or the\n    stop button). Interrupting the cell ends only the wait: ComfyUI goes on."""\n    running = paths.PRIVATE / "processes.json"\n    if not running.exists():\n        print(t("keep.not_running"))\n        return\n    show = show or rewritten_line()\n    shown = None\n    try:\n        while running.exists():\n            if shown is None or time.monotonic() - shown >= KEEP_RUNNING_REFRESH:\n                show(keep_running_status(running))\n                shown = time.monotonic()\n            time.sleep(KEEP_RUNNING_SECONDS)\n    except KeyboardInterrupt:\n        print(t("keep.interrupted"))\n        return\n    print(t("keep.stopped"))\n\n\ndef keep_running_status(running):\n    """One line: how long ComfyUI has run, and when the environment last reached Drive."""\n    minutes = int(time.time() - running.stat().st_mtime) // 60\n    status = paths.PRIVATE / "checkpoint-status.json"\n    try:\n        saved = json.loads(status.read_text())\n    except (OSError, ValueError):\n        saved = None\n    if saved is None:\n        last = t("keep.saved_none")\n    elif not saved.get("ok"):\n        last = t("keep.saved_failed")\n    else:\n        last = t("keep.saved_minutes", minutes=int(time.time() - status.stat().st_mtime) // 60)\n    return t("keep.status", running=f"{minutes // 60}:{minutes % 60:02d}", saved=last)\n\n\ndef rewritten_line():\n    """A function that shows a line in the cell\'s output in place of the one it showed before."""\n    try:\n        from IPython import get_ipython\n        from IPython.display import Pretty, display\n    except ImportError:\n        return print\n    if get_ipython() is None:\n        return print\n    handle = display(Pretty(""), display_id=True)\n    return lambda text: handle.update(Pretty(text))\n\n\ndef stop(config, save=True):\n    if save:\n        environment.checkpoint(config, force=True)  # A failed save must not look like successful shutdown.\n        print(t("stop.saved"))\n    file = paths.PRIVATE / "processes.json"\n    if file.exists():\n        for pid in reversed(list(json.loads(file.read_text()).values())):\n            try:\n                os.killpg(pid, signal.SIGTERM)\n            except ProcessLookupError:\n                pass\n        for pid in json.loads(file.read_text()).values():\n            deadline = time.monotonic() + 5\n            while time.monotonic() < deadline:\n                stat = Path(f"/proc/{pid}/stat")\n                if not stat.exists():\n                    break\n                try:\n                    if stat.read_text().rsplit(")", 1)[1].split()[0] == "Z":\n                        break\n                except FileNotFoundError:\n                    break\n                time.sleep(0.1)\n            else:\n                try:\n                    os.killpg(pid, signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n        file.unlink()\n    commands.stop_app_processes()\n    for watch in stop_button.STOP_WATCH:\n        watch.set()\n    for name in ("pairing.json", "stop-request.json", "stop-status.json"):\n        (paths.PRIVATE / name).unlink(missing_ok=True)\n    for name in SESSION_KEYS:\n        config.pop(name, None)\n    filesystem.private_write(paths.CONFIG, json.dumps(config))\n    print(t("stop.done"))\n\n\ndef zip_outputs(destination="/content/comfy-output.zip"):\n    try:\n        with open(destination, "wb") as file:\n            app_tree.app_action("zip-output", stdout=file, limit=1024 ** 4)\n    except Exception:\n        Path(destination).unlink(missing_ok=True)\n        raise\n    return str(destination)\n', 'stop_button.py': '"""The stop button the proxy adds to ComfyUI\'s page, and carrying out what it asks for."""\n\nimport json\nimport threading\n\nfrom . import drive, filesystem, paths, service\nfrom .messages import t\n\n\nSTOP_PATH = "/_colab/stop"\nSTOP_SCRIPT_PATH = "/_colab/stop.js"\n# Where the GitHub mark in the stop dialog leads. Empty: no mark.\nREPOSITORY_URL = "https://github.com/NeviumX/ComfyUI-instance-on-colab"\n# Added to the ComfyUI page by the proxy: a small button in the tab row, and the dialog it opens.\nSTOP_SCRIPT = r"""(() => {\n  if (window.top !== window || document.getElementById("colab-stop-button")) return;\n  const text = __TEXTS__;\n  const style = document.createElement("style");\n  style.textContent = `\n    #colab-stop-button { position: fixed; top: 7px; right: 56px; z-index: 1000; font: 12px/1 system-ui, sans-serif;\n      padding: 5px 10px; border-radius: 6px; border: 1px solid rgba(128, 128, 128, .5); background: rgba(30, 30, 30, .7);\n      color: #ddd; cursor: pointer; opacity: .6; }\n    #colab-stop-button:hover, #colab-stop-button:focus-visible { opacity: 1; color: #fff; border-color: #f87171; }\n    #colab-stop-dialog { font: 14px/1.6 system-ui, sans-serif; width: min(26rem, calc(100vw - 2rem)); padding: 1.25rem;\n      border: 1px solid #555; border-radius: 10px; background: #202020; color: #eee;\n      box-shadow: 0 12px 40px rgba(0, 0, 0, .6); }\n    #colab-stop-dialog::backdrop { background: rgba(0, 0, 0, .6); }\n    #colab-stop-dialog h2 { margin: 0 2.25rem .5rem 0; font-size: 1.1rem; }\n    #colab-stop-dialog .colab-stop-repository { position: absolute; top: 1.1rem; right: 1.1rem; display: flex; color: #aaa; }\n    #colab-stop-dialog .colab-stop-repository:hover, #colab-stop-dialog .colab-stop-repository:focus-visible { color: #fff; }\n    #colab-stop-dialog p { margin: 0 0 .75rem; }\n    #colab-stop-dialog .colab-stop-status { color: #fcd34d; }\n    #colab-stop-dialog .colab-stop-status:empty { display: none; }\n    #colab-stop-dialog .colab-stop-actions { display: flex; flex-direction: column; gap: .5rem; }\n    #colab-stop-dialog button { font: inherit; padding: .55rem .75rem; border-radius: 6px; border: 1px solid #666;\n      background: #333; color: #eee; cursor: pointer; }\n    #colab-stop-dialog button:hover:enabled { background: #444; }\n    #colab-stop-dialog button:disabled { opacity: .45; cursor: default; }\n    #colab-stop-dialog button.colab-stop-danger { background: #b91c1c; border-color: #b91c1c; color: #fff; }\n    #colab-stop-dialog button.colab-stop-danger:hover:enabled { background: #dc2626; }\n    #colab-stop-dialog button.colab-stop-quiet { background: transparent; }\n  `;\n  const add = (parent, tag, content, name) => {\n    const element = document.createElement(tag);\n    if (content) element.textContent = content;\n    if (name) element.className = name;\n    if (tag === "button") element.type = "button";\n    return parent.appendChild(element);\n  };\n  const dialog = document.createElement("dialog");\n  dialog.id = "colab-stop-dialog";\n  add(dialog, "h2", text.title);\n  add(dialog, "p", text.explain);\n  const status = add(dialog, "p", "", "colab-stop-status");\n  const actions = add(dialog, "div", "", "colab-stop-actions");\n  const remove = add(actions, "button", text.delete, "colab-stop-danger");\n  const disconnect = add(actions, "button", text.disconnect);\n  const cancel = add(actions, "button", text.cancel, "colab-stop-quiet");\n  cancel.autofocus = true;  // Enter right after opening must not be a choice to stop.\n  const repository = "__REPOSITORY__";\n  if (repository) {\n    // Last in the dialog, so the keyboard reaches the choices first; shown in its top right corner.\n    const link = add(dialog, "a", "", "colab-stop-repository");\n    link.href = repository;\n    link.target = "_blank";\n    link.rel = "noopener noreferrer";\n    link.title = text.repository;\n    link.setAttribute("aria-label", text.repository);\n    const icon = document.createElementNS("http://www.w3.org/2000/svg", "svg");\n    for (const [name, value] of [["viewBox", "0 0 16 16"], ["width", "20"], ["height", "20"], ["fill", "currentColor"], ["aria-hidden", "true"]]) {\n      icon.setAttribute(name, value);\n    }\n    const mark = document.createElementNS("http://www.w3.org/2000/svg", "path");\n    mark.setAttribute("d", "M8 0C3.58 0 0 3.58 0 8c0 3.54 2.29 6.53 5.47 7.59.4.07.55-.17.55-.38 0-.19-.01-.82-.01-1.49-2.01.37-2.53-.49-2.69-.94-.09-.23-.48-.94-.82-1.13-.28-.15-.68-.52-.01-.53.63-.01 1.08.58 1.23.82.72 1.21 1.87.87 2.33.66.07-.52.28-.87.51-1.07-1.78-.2-3.64-.89-3.64-3.95 0-.87.31-1.59.82-2.15-.08-.2-.36-1.02.08-2.12 0 0 .67-.21 2.2.82.64-.18 1.32-.27 2-.27.68 0 1.36.09 2 .27 1.53-1.04 2.2-.82 2.2-.82.44 1.1.16 1.92.08 2.12.51.56.82 1.27.82 2.15 0 3.07-1.87 3.75-3.65 3.95.29.25.54.73.54 1.48 0 1.07-.01 1.93-.01 2.2 0 .21.15.46.55.38A8.013 8.013 0 0016 8c0-4.42-3.58-8-8-8z");\n    icon.appendChild(mark);\n    link.appendChild(icon);\n  }\n  const button = document.createElement("button");\n  button.id = "colab-stop-button";\n  button.type = "button";\n  button.textContent = text.stop;\n  button.title = text.title;\n  let busy = false;\n  const lock = (locked) => {\n    busy = locked;\n    for (const choice of [remove, disconnect, cancel]) choice.disabled = locked;\n  };\n  const ask = async (deleteRuntime) => {\n    lock(true);\n    status.textContent = text.working;\n    try {\n      const answer = await fetch("__STOP_PATH__", {method: "POST", headers: {"Content-Type": "application/json"},\n                                               body: JSON.stringify({delete_runtime: deleteRuntime})});\n      if (!answer.ok) throw new Error(answer.status);\n    } catch (error) {\n      status.textContent = text.failed;\n      lock(false);\n      return;\n    }\n    const started = Date.now();\n    const timer = setInterval(async () => {\n      let state;\n      try {\n        const answer = await fetch("__STOP_PATH__", {cache: "no-store"});\n        if (!answer.ok) throw new Error(answer.status);\n        state = (await answer.json()).state;\n      } catch (error) {\n        // Nothing answers any more: the proxy and the tunnel are down.\n        clearInterval(timer);\n        status.textContent = deleteRuntime ? text.deleted : text.stopped;\n        button.remove();\n        return;\n      }\n      if (state === "failed") {\n        clearInterval(timer);\n        status.textContent = text.failed;\n        lock(false);\n      } else if (Date.now() - started > 180000) {\n        status.textContent = text.slow;\n      }\n    }, 3000);\n  };\n  remove.addEventListener("click", () => ask(true));\n  disconnect.addEventListener("click", () => ask(false));\n  cancel.addEventListener("click", () => dialog.close());\n  dialog.addEventListener("cancel", (event) => { if (busy) event.preventDefault(); });\n  // Keys pressed in the dialog are not ComfyUI shortcuts.\n  dialog.addEventListener("keydown", (event) => event.stopPropagation());\n  button.addEventListener("click", () => {\n    if (!busy) status.textContent = "";\n    dialog.showModal();\n  });\n  const mount = () => {\n    document.head.appendChild(style);\n    document.body.append(button, dialog);\n  };\n  if (document.body) mount(); else document.addEventListener("DOMContentLoaded", mount);\n})();\n"""\n\n\ndef with_stop_button(page):\n    """The ComfyUI page with the script that adds the stop button (unchanged if it has no head)."""\n    position = page.lower().rfind(b"</head>")\n    tag = b\'<script defer src="\' + STOP_SCRIPT_PATH.encode() + b\'"></script>\'\n    return page if position < 0 else page[:position] + tag + page[position:]\n\n\ndef stop_button_script():\n    texts = {"stop": t("menu.stop"), "title": t("menu.title"), "explain": t("menu.explain"), "delete": t("menu.delete"),\n             "disconnect": t("menu.disconnect"), "cancel": t("menu.cancel"), "working": t("menu.working"),\n             "stopped": t("menu.stopped"), "deleted": t("menu.deleted"), "failed": t("menu.failed"), "slow": t("menu.slow"),\n             "repository": t("menu.repository")}\n    repository = REPOSITORY_URL if REPOSITORY_URL.startswith("https://") else ""\n    return (STOP_SCRIPT.replace("__TEXTS__", json.dumps(texts)).replace("__STOP_PATH__", STOP_PATH)\n            .replace(\'"__REPOSITORY__"\', json.dumps(repository)))\n\n\ndef request_stop(delete_runtime):\n    """Leave word, from the proxy, that the ComfyUI page asked to stop. The notebook\'s own\n    process carries it out: only it can save to Drive and give the runtime back to Colab."""\n    (paths.PRIVATE / "stop-status.json").unlink(missing_ok=True)\n    filesystem.private_write(paths.PRIVATE / "stop-request.json", json.dumps({"delete_runtime": bool(delete_runtime)}))\n\n\ndef stop_request_state():\n    if (paths.PRIVATE / "stop-request.json").exists():\n        return "pending"\n    return "failed" if (paths.PRIVATE / "stop-status.json").exists() else "idle"\n\n\ndef delete_colab_runtime():\n    """Finish writing to Drive, then give the runtime back: it stops using compute units."""\n    from google.colab import drive as colab_drive, runtime as colab\n    drive.hide_drive_models()\n    colab_drive.flush_and_unmount()\n    colab.unassign()\n\n\ndef carry_out_stop_request(config):\n    """Do what the stop button of the ComfyUI page asked for, if it asked. True once stopped."""\n    request = paths.PRIVATE / "stop-request.json"\n    try:\n        wanted = json.loads(request.read_text())\n    except (OSError, ValueError):\n        return False\n    try:\n        service.stop(config, save=True)\n    except Exception as failure:\n        # Nothing was stopped, so the page is still there to say that it did not work.\n        filesystem.private_write(paths.PRIVATE / "stop-status.json", json.dumps({"state": "failed"}))\n        request.unlink(missing_ok=True)\n        print(t("menu.failed"), failure, flush=True)\n        return False\n    if wanted.get("delete_runtime") is True:\n        delete_colab_runtime()\n    return True\n\n\nSTOP_WATCH = []\nSTOP_WATCH_SECONDS = 2\n\n\ndef follow_stop_requests(config):\n    """Watch for the stop button from the notebook\'s process, until stopped one way or another."""\n    cancelled = threading.Event()\n    STOP_WATCH[:] = [cancelled]\n\n    def watch():\n        while not cancelled.wait(STOP_WATCH_SECONDS):\n            if carry_out_stop_request(config):\n                return\n    threading.Thread(target=watch, daemon=True).start()\n'}
package = Path('/content/comfy_colab_runtime')
shutil.rmtree(package, ignore_errors=True)
package.mkdir()
for name, text in FILES.items():
    (package / name).write_text(text, encoding="utf-8")
Path('/content/comfy_colab_runtime.py').unlink(missing_ok=True)  # Written by earlier versions of this notebook.
# Run again: import the code just written, not the one imported before.
for name in [name for name in sys.modules if name.split(".")[0] == 'comfy_colab_runtime']:
    del sys.modules[name]
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
importlib.invalidate_caches()
import comfy_colab_runtime as runtime
print("运行环境代码已解包。")


In [ ]:
#@title 3. 连接 Google Drive 并安装 / 恢复环境
#@markdown 在弹出的 Google 授权界面允许访问 Google Drive。初始化可能需要几分钟。
from IPython.display import Javascript, display
display(Javascript("google.colab.output.setIframeHeight(0, true, {maxHeight: 300})"))
runtime.setup(root_name=DRIVE_FOLDER_NAME, comfy_ref=COMFY_REF,
              checkpoint_seconds=CHECKPOINT_SECONDS, apt_packages=APT_PACKAGES.split(),
              restore=RESTORE_SAVED_ENVIRONMENT, model_access=MODEL_ACCESS,
              sage_attention=USE_SAGE_ATTENTION, fp16_accumulation=USE_FP16_ACCUMULATION)
print("模型请放在：我的云端硬盘/" + DRIVE_FOLDER_NAME + "/models/")


## 模型管理

Google Drive 中的 `ComfyUI-Colab/models/` 按模型类型分为 `checkpoints`、`diffusion_models`、`text_encoders`、`vae`、`loras`、`controlnet` 等目录。

- **手动上传：** 将模型上传到 Drive 中相应目录，然后执行第 5 步同步。
- **通过链接下载：** 在第 4 步填写 Hugging Face 或 Civitai 的模型链接并执行。默认下载到 Drive，之后可重复使用；启用 `RUNTIME_ONLY` 时只保存到本次 Colab 运行时，运行时删除后模型也会消失。
- **加载方式：** `MODEL_ACCESS=direct` 按需从 Drive 读取，首次加载模型较慢；`MODEL_ACCESS=copy` 在初始化时将全部模型复制到 Colab 本地磁盘，需要预留足够空间。


In [ ]:
#@title 4. 通过链接下载模型（Hugging Face / Civitai）
#@markdown 粘贴 Hugging Face 的 Files and versions 文件页面链接，或 Civitai 已选择模型版本的页面 / 下载链接。
MODEL_URL = "" #@param {type:"string"}
#@markdown 模型保存目录（也可手动输入列表中没有的目录名称）。
MODEL_FOLDER = "checkpoints" #@param ["checkpoints", "diffusion_models", "text_encoders", "vae", "loras", "controlnet", "clip_vision", "upscale_models", "embeddings"] {allow-input: true}
#@markdown 保存文件名（留空则沿用网站提供的名称）。
MODEL_FILENAME = "" #@param {type:"string"}
#@markdown 网站公布的 SHA256 校验值（可选；填写后将校验文件）。
MODEL_SHA256 = "" #@param {type:"string"}
#@markdown 需要登录才能下载时，在 Colab「密钥」（钥匙图标）中添加 `HF_TOKEN` 或 `CIVITAI_TOKEN`，再勾选此项。
USE_DOWNLOAD_TOKEN = False #@param {type:"boolean"}
#@markdown 只在本次运行时使用，不保存到 Google Drive；删除运行时后模型会消失。
RUNTIME_ONLY = False #@param {type:"boolean"}
DOWNLOAD_NOW = False #@param {type:"boolean"}
if DOWNLOAD_NOW:
    config = runtime.load_config()
    token = ""
    try:
        if USE_DOWNLOAD_TOKEN:
            from google.colab import userdata
            token = userdata.get(runtime.model_token_name(MODEL_URL))
        runtime.fetch_model(config, MODEL_URL, MODEL_FOLDER, MODEL_FILENAME, MODEL_SHA256, token,
                            runtime_only=RUNTIME_ONLY)
    finally:
        token = ""
else:
    print("如需下载模型，请输入 URL、勾选 DOWNLOAD_NOW，再运行此单元格。")


In [ ]:
#@title 5. 同步手动上传到 Google Drive 的模型
REFRESH_MODELS = False #@param {type:"boolean"}
if REFRESH_MODELS:
    config = runtime.load_config()
    runtime.sync_models(config)
    print("如已连接 ComfyUI，请刷新页面。如果替换了同名模型，请先通过第 10 步停止（关闭 FLUSH_AND_UNMOUNT），再运行 6-1、6-2 或 6-3。")
else:
    print("手动向 Google Drive 添加模型后，请勾选 REFRESH_MODELS 并运行。")


## 启动与连接

从以下三种方式中选择一种并运行对应单元格。切换连接方式前先通过第 10 步停止，并关闭 `FLUSH_AND_UNMOUNT`，以保留当前运行时的 Drive 挂载。首次打开界面可能需要约一分钟。

- **6-1 私有 URL：** 适合在打开此 Notebook 的浏览器中使用，只有登录了运行此 Notebook 的 Google 账号的浏览器可以访问。不对外公开，Comfy Desktop 无法使用这种连接方式。
- **6-2 公共 URL：** 适合在其他设备或 Comfy Desktop 的 **Remote Connection（远程连接）** 中使用。打开显示的 URL 后输入 Notebook 中的配对码即可连接；每个配对码只能使用一次，有效期 10 分钟。执行第 10 步停止后 URL 失效。
- **6-3 自定义域名：** 使用自己的固定域名，通过 Cloudflare Access 登录，仅允许事先授权的账号访问。需要提前准备 Cloudflare 账号、域名并完成配置。
- 在 Comfy Desktop 的连接设置中启用 **Auto-download outputs（自动下载输出）**，可将生成文件保存到电脑。通过浏览器使用或遗漏的输出可在第 9 步打包下载。
- 启动单元格会持续运行，以降低只操作 ComfyUI 页面时被 Colab 判断为空闲的概率。需要运行其他单元格时先点 ■ 中断等待，完成后运行 **6-4** 恢复等待。
- 如果界面打不开，在第 7 步启用连接诊断。
- ComfyUI 页面顶部的 **Stop（停止）** 按钮也可以保存环境并断开连接，可选择同时删除运行时。


In [ ]:
#@title 6-1. 启动并连接：私有 URL（仅当前浏览器）
config = runtime.load_config()
runtime.start(config, "colab")
runtime.keep_running()


In [ ]:
#@title 6-2. 启动并连接：公共 URL
#@markdown 6-2 启动后，如需添加其他设备或配对码已过期，请勾选此项再运行以生成新配对码。
NEW_PAIRING_CODE = False #@param {type:"boolean"}
config = runtime.load_config()
if NEW_PAIRING_CODE:
    runtime.issue_pairing_code(config)
else:
    runtime.start(config, "tunnel")
runtime.keep_running()


In [ ]:
#@title 6-3. 启动并连接：自定义域名
#@markdown 先在 Cloudflare Zero Trust 创建命名隧道（公开主机名指向 `http://127.0.0.1:8189`），并创建保护该主机名的 Access 应用。在 Colab「密钥」中分别将以 `eyJ` 开头的隧道令牌保存为 `CF_TUNNEL_TOKEN`、Access 应用的 64 字符 Application Audience (AUD) Tag 保存为 `CF_ACCESS_AUD`，并授权此 Notebook 读取。
#@markdown
#@markdown 隧道的公开主机名，例如 `comfy.example.com`。
ACCESS_HOSTNAME = "" #@param {type:"string"}
config = runtime.load_config()
from google.colab import userdata
token, audience = userdata.get("CF_TUNNEL_TOKEN"), userdata.get("CF_ACCESS_AUD")
try:
    runtime.start_with_access(config, ACCESS_HOSTNAME, audience, token)
finally:
    token = audience = ""
runtime.keep_running()


In [ ]:
#@title 6-4. 恢复等待（降低空闲断开概率）
#@markdown 6-1 至 6-3 启动 ComfyUI 后会保持单元格运行，以降低仅操作 ComfyUI 页面时被 Colab 判为空闲的概率。若要执行其他单元格，先点击 ■ 中断等待，完成操作后运行这里继续等待。中断等待不会停止 ComfyUI。请保持 Colab 标签页打开。
runtime.keep_running()


In [ ]:
#@title 7. 查看保存状态、日志及连接诊断
#@markdown 如果界面打不开，勾选并运行以检查连接卡在哪个环节。
RUN_CONNECTION_DIAGNOSIS = False #@param {type:"boolean"}
if RUN_CONNECTION_DIAGNOSIS:
    runtime.run_diagnosis()
path = runtime.paths.PRIVATE / "checkpoint-status.json"
print(path.read_text() if path.exists() else "目前还没有保存状态。")
for name in ("comfy.log", "supervisor.log", "proxy.log", "tunnel.log"):
    path = runtime.paths.PRIVATE / name
    if path.exists():
        print("---", name)
        print(runtime.redact(path.read_text(errors="replace")[-5000:]))


## 保存与结束

ComfyUI 程序、自定义节点和设置会自动保存到 Google Drive，并在下次执行第 3 步时恢复。想立即保存最新更改可运行第 8 步。**生成的图片和其他输出文件不会自动保存到 Google Drive。**

结束使用前，请先把生成结果保存到本地电脑，再执行第 10 步，确认停止和保存操作完成后再删除 Colab 运行时。


In [ ]:
#@title 8. 立即将运行环境保存到 Google Drive
SAVE_NOW = False #@param {type:"boolean"}
if SAVE_NOW:
    config = runtime.load_config()
    runtime.checkpoint(config, force=True)
    print("已将环境和新模型保存到 Google Drive。")
else:
    print("需要保存时勾选 SAVE_NOW 并运行。")


In [ ]:
#@title 9. 将生成文件打包为 ZIP 下载到电脑
DOWNLOAD_OUTPUT_ZIP = False #@param {type:"boolean"}
if DOWNLOAD_OUTPUT_ZIP:
    from google.colab import files
    files.download(runtime.zip_outputs())
else:
    print("需要打包下载时勾选 DOWNLOAD_OUTPUT_ZIP 并运行。")


In [ ]:
#@title 10. 保存并停止
STOP_NOW = False #@param {type:"boolean"}
#@markdown 完成 Google Drive 写入并卸载云盘（删除运行时前应勾选；如果要在当前运行时重新执行 6-1 至 6-3，则关闭）。
FLUSH_AND_UNMOUNT = True #@param {type:"boolean"}
if STOP_NOW:
    config = runtime.load_config()
    runtime.stop(config, save=True)
    if FLUSH_AND_UNMOUNT:
        from google.colab import drive
        runtime.hide_drive_models()
        drive.flush_and_unmount()
        print("Google Drive 写入和卸载已完成，现在可以删除运行时。")
    else:
        print("Google Drive 仍处于挂载状态。删除运行时前，请勾选 FLUSH_AND_UNMOUNT 再执行此单元格。")
else:
    print("如需结束，请勾选 STOP_NOW 并运行。")


## 更新 ComfyUI

恢复已有环境时，会继续使用保存时的 ComfyUI 版本。需要保留自定义节点、设置和工作流并更新 ComfyUI 本体时，运行第 11 步。

- **操作顺序：** 勾选第 0 步检查最新版本（或在第 1 步的 `COMFY_REF` 填写目标版本）→ 1 → 2 → 3 → 11 → 6-1/6-2/6-3。如果程序已经启动，先在第 10 步停止并关闭 `FLUSH_AND_UNMOUNT`，然后依次执行 0 → 1 → 11 → 6-1/6-2/6-3。
- 更新后的环境会保存到 Google Drive，以后默认恢复该版本。
- 如果更新失败（例如某个自定义节点不兼容），会尝试恢复到之前的 ComfyUI 版本，Google Drive 中的已保存环境保持不变。


In [ ]:
#@title 11. 更新 ComfyUI（保留自定义节点与设置）
#@markdown 更新到 `COMFY_REF` 对应版本（若第 0 步已选择最新版本则使用该版本）。请在 ComfyUI 停止时执行。
UPDATE_COMFYUI_NOW = False #@param {type:"boolean"}
if UPDATE_COMFYUI_NOW:
    config = runtime.load_config()
    runtime.update_comfyui(config, COMFY_REF)
else:
    print("如需更新，勾选 UPDATE_COMFYUI_NOW 后运行。")


## 安全说明

- 6-1 不向公网开放。6-2 和 6-3 只提供需要身份验证的入口：6-2 要求设备输入配对码，6-3 要求通过 Cloudflare Access 登录。
- 自定义节点会在运行时中执行自己的代码。此 Notebook 会对 Google Drive 和连接密钥采取隔离措施，但并非完全隔离；请只安装可信来源的节点和模型。
- 任何成功连接的用户都可以安装自定义节点，保存到环境中的节点下次还会运行。如果安装了不可信的节点，请关闭第 1 步中的 `RESTORE_SAVED_ENVIRONMENT`，重新建立干净的环境。
- 不要点击 Colab 自带的「挂载云端硬盘」按钮；请仅通过本 Notebook 的第 3 步连接 Google Drive。
